<a href="https://colab.research.google.com/github/WardaAli-00/MS-THESIS-/blob/main/THESIS_IMPLEMENTATION_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### (Baseline): Text-Only Claim Verification Pipeline

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install transformers torch scikit-learn pandas tqdm newspaper3k lxml_html_clean google-search-results

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 75.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 12.4 MB/s eta 0:00:00


In [3]:
import os
import re
import torch
from newspaper import Article
from serpapi import GoogleSearch
from transformers import pipeline

In [4]:
device_id = 0 if torch.cuda.is_available() else -1
print("device_id:", device_id)

device_id: 0


In [5]:
!pip uninstall -y serpapi google-search-results
!pip install google-search-results==2.4.2

Found existing installation: google_search_results 2.4.2
Uninstalling google_search_results-2.4.2:
  Successfully uninstalled google_search_results-2.4.2
  Using cached google_search_results-2.4.2-py3-none-any.whl


In [77]:
SERPAPI_KEY = "a88883d9b8af70f3ba224fa4209127e2eb9fd800e477353f991bc6c425c65c80"

In [78]:
import os
os.environ["SERPAPI_KEY"] = "a88883d9b8af70f3ba224fa4209127e2eb9fd800e477353f991bc6c425c65c80"
SERPAPI_KEY = os.getenv("SERPAPI_KEY")

In [8]:
print("SERPAPI_KEY exists:", bool(SERPAPI_KEY))
print("SERPAPI_KEY length:", len(SERPAPI_KEY) if SERPAPI_KEY else 0)

SERPAPI_KEY exists: True
SERPAPI_KEY length: 64


In [9]:
from serpapi import GoogleSearch

params = {
    "engine": "google",
    "q": "NASA Mars water",
    "api_key": SERPAPI_KEY,
    "num": 2
}
results = GoogleSearch(params).get_dict()
print(results.keys())
print("error:", results.get("error"))
print("organic_results_count:", len(results.get("organic_results", [])))

dict_keys(['error'])
error: Your account has run out of searches.
organic_results_count: 0


In [ ]:
nli = pipeline(
    "text-classification",
    model="typeform/distilbert-base-uncased-mnli",
    return_all_scores=True,
    device=device_id
)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

 RETRIEVAL MODULE

In [ ]:
def is_valid_url(url: str) -> bool:
    bad_ext = (".jpg", ".jpeg", ".png", ".gif", ".webp", ".mp4", ".mp3", ".pdf")
    u = url.lower()
    if u.endswith(bad_ext):
        return False
    if "image" in u:
        return False
    return True

In [ ]:

def fetch_article_text(url: str, min_chars: int = 200, max_chars: int = 1200):
    try:
        article = Article(url)
        article.download()
        article.parse()
        text = " ".join(article.text.split())
        if len(text) < min_chars:
            return None
        return text[:max_chars]
    except Exception:
        return None

In [ ]:
def retrieve_evidence_serpapi(query: str, max_results: int = 5):
    params = {
        "engine": "google",
        "q": query,
        "api_key": SERPAPI_KEY,
        "num": max_results
    }
    results = GoogleSearch(params).get_dict()
    evidence = []
    for r in results.get("organic_results", []):
        url = r.get("link", "")
        snippet = r.get("snippet", "")
        if not url or not is_valid_url(url):
            continue
        text = fetch_article_text(url)
        # fallback to snippet if article extraction fails
        if not text and snippet and len(snippet) > 80:
            text = snippet
        if text:
            evidence.append({"text": text, "url": url})
    return evidence

CLEANING + RANKING

In [ ]:
def is_valid_evidence(text: str) -> bool:
    noise_keywords = [
        "home", "all news", "media inquiries", "campus", "podcasts", "advertisement"
    ]
    t = text.lower()
    if len(text) < 120:
        return False
    if any(k in t for k in noise_keywords):
        return False
    return True

In [ ]:
def deduplicate_evidence(evidence_list):
    seen = set()
    out = []
    for ev in evidence_list:
        key = (ev["url"], ev["text"][:200])
        if key not in seen:
            out.append(ev)
            seen.add(key)
    return out

In [ ]:
def clean_evidence(evidence_list):
    cleaned = []
    for ev in evidence_list:
        text = " ".join(ev["text"].split())
        if is_valid_evidence(text):
            cleaned.append({"text": text, "url": ev["url"]})
    return deduplicate_evidence(cleaned)

In [ ]:
def rank_by_overlap(claim: str, evidence_list):
    claim_words = set(re.findall(r"\w+", claim.lower()))
    scored = []
    for ev in evidence_list:
        ev_words = set(re.findall(r"\w+", ev["text"].lower()))
        overlap = len(claim_words.intersection(ev_words))
        scored.append((overlap, ev))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [ev for _, ev in scored]

REASONING MODULE (NLI)

In [ ]:
 def nli_text_score(claim: str, evidence_list, top_k: int = 2):
    if not evidence_list:
        return {"entail": 0.0, "contra": 0.0, "neutral": 1.0, "used": 0}

    e_sum = c_sum = n_sum = 0.0
    used = 0

    for ev in evidence_list[:top_k]:
        text = ev["text"].strip()
        if len(text) < 20:
            continue

        try:
            out = nli(
                {"text": text[:400], "text_pair": claim},
                top_k=None      # ← this is the key fix
            )

            # Normalise shape
            if isinstance(out, list) and len(out) > 0:
                if isinstance(out[0], list):
                    out = out[0]

            scores = {}
            for d in out:
                if isinstance(d, dict) and "label" in d:
                    scores[d["label"].upper()] = float(d.get("score", 0.0))

            if len(scores) < 2:
                print(f"  ⚠️  Still incomplete NLI output: {scores}")
                continue

            e_sum += scores.get("ENTAILMENT",   0.0)
            c_sum += scores.get("CONTRADICTION", 0.0)
            n_sum += scores.get("NEUTRAL",       0.0)
            used  += 1

            print(f"  NLI: entail={scores.get('ENTAILMENT',0):.3f}  "
                  f"contra={scores.get('CONTRADICTION',0):.3f}  "
                  f"neutral={scores.get('NEUTRAL',0):.3f}")

        except Exception as ex:
            print(f"  NLI ERROR: {ex}")
            continue

    if used == 0:
        return {"entail": 0.0, "contra": 0.0, "neutral": 1.0, "used": 0}

    return {
        "entail":  round(e_sum / used, 4),
        "contra":  round(c_sum / used, 4),
        "neutral": round(n_sum / used, 4),
        "used":    used,
    }

EXPLANATION + CONFIDENCE

In [ ]:
def final_text_label(text_scores, margin=0.03, contra_floor=0.08, entail_floor=0.12):
    e = text_scores["entail"]
    c = text_scores["contra"]
    n = text_scores["neutral"]
    if c >= contra_floor and (c - e) >= margin:
        return "FAKE"
    elif e >= entail_floor and (e - c) >= margin:
        return "REAL"
    else:
        return "MISLEADING"

In [ ]:
def compute_confidence(nli_stats, pred_label):
    if nli_stats["used"] == 0:
        return 0.0

    e = nli_stats["entail"]
    c = nli_stats["contra"]
    n = nli_stats["neutral"]

    if pred_label == "REAL":
        return round(e * 100, 2)
    elif pred_label == "FAKE":
        return round(c * 100, 2)
    else:
        uncertainty = 1 - abs(e - c)
        return round(uncertainty * 100, 2)

In [ ]:
def generate_explanation(claim, evidence_list, label):
    if not evidence_list:
        return "No strong evidence found."
    msg = [f"Claim: {claim}", "", "Top evidence:"]
    for i, ev in enumerate(evidence_list[:2], start=1):
        msg.append(f"{i}. {ev['text'][:180]}...")
        msg.append(f"   Source: {ev['url']}")
    if label == "REAL":
        msg.append("Verdict rationale: Most evidence supports the claim.")
    elif label == "FAKE":
        msg.append("Verdict rationale: Most evidence contradicts the claim.")
    else:
        msg.append("Verdict rationale: Evidence is mixed/insufficient.")
    return "\n".join(msg)

 MAIN PIPELINE (Methodology Core)

In [ ]:
def main_pipeline(claim: str, top_k: int = 2):
    retrieved = retrieve_evidence_serpapi(claim, max_results=6)

    cleaned = clean_evidence(retrieved)
    ranked = rank_by_overlap(claim, cleaned)
    top_evidence = ranked[:top_k]
    nli_stats  = nli_text_score(claim, top_evidence, top_k)
    pred_label = final_text_label(nli_stats)
    confidence = compute_confidence(nli_stats, pred_label)
    explanation = generate_explanation(claim, top_evidence, pred_label)
    return {
        "claim": claim,
        "label": pred_label,
        "confidence": confidence,
        "nli_stats": nli_stats,
        "evidence": top_evidence,
        "explanation": explanation
    }
# quick sanity run
out = main_pipeline("NASA confirms water on Mars")
print(out["label"], out["confidence"])
print(out["explanation"][:500])

MISLEADING 100.0
Claim: NASA confirms water on Mars

Top evidence:
1. These dark, narrow, 100 meter-long streaks called recurring slope lineae flowing downhill on Mars are inferred to have been formed by contemporary flowing water. Recently, planetar...
   Source: https://www.nasa.gov/news-release/nasa-confirms-evidence-that-liquid-water-flows-on-todays-mars/
2. Using seismic activity to probe the interior of Mars, geophysicists have found evidence for a large underground reservoir of liquid water — enough to fi


### V2: MULTIMODAL VISION-RAG PIPELINE (SKELETON)

In [10]:
!pip install -q beautifulsoup4
from bs4 import BeautifulSoup

In [11]:
!pip install -q open_clip_torch sentence-transformers newspaper3k lxml_html_clean

import re
import os
import time
import torch
import requests
from PIL import Image
from io import BytesIO
from newspaper import Article
from serpapi import GoogleSearch
from transformers import pipeline as hf_pipeline
from sentence_transformers import SentenceTransformer
import open_clip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00


NLI model

In [12]:
nli = hf_pipeline(
    "text-classification",
    model="MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
    return_all_scores=True,
    device=device_id,
    truncation=True,
    max_length=512,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/395 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Sentence Transformer

In [13]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Models loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models loaded.


CLIP model

In [14]:
import open_clip
import torch
from PIL import Image

In [15]:
import torch

device_id = 0 if torch.cuda.is_available() else -1
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
HEADERS   = {"User-Agent": "Mozilla/5.0"}

In [16]:
import torch
import open_clip
from PIL import Image

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai"
)
clip_model = clip_model.to(DEVICE)
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")

HEADERS = {"User-Agent": "Mozilla/5.0"}

open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


In [17]:
def compute_clip_score(image: Image.Image, text: str) -> float:
    try:
        image_input = clip_preprocess(image).unsqueeze(0).to(DEVICE)
        text_input  = clip_tokenizer([text]).to(DEVICE)
        with torch.no_grad():
            img_feat = clip_model.encode_image(image_input)
            txt_feat = clip_model.encode_text(text_input)
            img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
            txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
            score    = (img_feat @ txt_feat.T).item()
        return float((score + 1) / 2)
    except Exception:
        return 0.0

DOMAINS

In [18]:
TRUSTED_DOMAINS = {
    "nasa.gov": 3.0, "nature.com": 3.0, "who.int": 3.0,
    "cdc.gov": 3.0, "nih.gov": 3.0, "sciencedirect.com": 2.8,
    "pubmed.ncbi.nlm.nih.gov": 3.0, "snopes.com": 2.8,
    "politifact.com": 2.8, "factcheck.org": 2.8, "fullfact.org": 2.8,
    "apnews.com": 2.5, "reuters.com": 2.5, "bbc.com": 2.0,
    "bbc.co.uk": 2.0, "nytimes.com": 1.8, "theguardian.com": 1.8,
    "scientificamerican.com": 2.2,
}

FACT_CHECK_DOMAINS = {
    "snopes.com", "politifact.com", "factcheck.org",
    "fullfact.org", "checkyourfact.com", "leadstories.com",
}

NOISE_DOMAINS = {
    "facebook.com", "instagram.com", "tiktok.com",
    "twitter.com", "x.com", "reddit.com",
    "freerepublic.com", "substack.com",
}

SCIENCE_KEYWORDS = {
    "boils", "melts", "freezes", "revolves", "orbits", "speed of light",
    "photosynthesis", "gravity", "electrons", "tectonic", "ozone layer",
    "heart pumps", "blood carries", "bones", "insulin regulates",
    "brain controls", "dna carries", "water cycle",
}

STOPWORDS = {
    "the","a","an","is","are","was","were","of","on","in","at","to",
    "for","and","or","not","do","does","can","it","that","this","be",
    "by","from","with","will","has","have","its","their","about",
    "also","but","so","if","as",
}

In [19]:
def get_domain(url):
    try:    return url.split("/")[2].replace("www.", "")
    except: return ""

Tweet claim cleaner

In [20]:
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def clean_tweet_claim(claim: str) -> str:
    claim = re.sub(r'http\S+', '', claim)
    claim = re.sub(r'[@#]\w+', '', claim)
    claim = re.sub(r'^RT\s+', '', claim)
    claim = re.sub(r'\s+', ' ', claim).strip()
    return claim if len(claim) > 15 else claim

In [21]:
def get_domain_boost(url: str) -> float:
    try:
        domain = url.split("/")[2].replace("www.", "")
        return TRUSTED_DOMAINS.get(domain, 0.0)
    except Exception:
        return 0.0


In [22]:
def get_domain_for_check(url: str) -> str:
    try:
        domain = url.split("/")[2].replace("www.", "")
        return domain
    except Exception:
        return ""

def is_fact_checker(url: str) -> bool:
    domain = get_domain_for_check(url)
    return domain in FACT_CHECK_DOMAINS

In [23]:
def is_health_fact(query):
    health_kws = {"causes","cancer","risk","disease","infection","bacteria",
                  "virus","vaccine","antibiotic","lung","smoking","alcohol",
                  "sugar","exercise","sleep"}
    return len(health_kws & set(re.findall(r"\w+", query.lower()))) >= 2

Embedding fallback

In [24]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def embedding_stance_fallback(claim, evidence_list):
    if not evidence_list:
        return {"entail": 0.10, "contra": 0.05, "neutral": 0.85, "used": 0}
    claim_emb = embed_model.encode([claim])
    sims = [float(cosine_similarity(claim_emb, embed_model.encode([ev["text"][:300]]))[0][0])
            for ev in evidence_list[:4]]
    avg = float(np.mean(sims)) if sims else 0.5
    if avg > 0.55:   return {"entail": 0.35, "contra": 0.05, "neutral": 0.60, "used": -1}
    elif avg < 0.30: return {"entail": 0.05, "contra": 0.35, "neutral": 0.60, "used": -1}
    else:            return {"entail": 0.20, "contra": 0.15, "neutral": 0.65, "used": -1}

In [53]:
def embedding_stance_fallback(claim: str, evidence_list: list) -> dict:
    if not evidence_list:
        return {"entail": 0.1, "contra": 0.05, "neutral": 0.85, "used": 0}

    claim_emb = embed_model.encode([claim])
    sims = []
    for ev in evidence_list[:4]:
        ev_emb = embed_model.encode([ev["text"][:300]])
        sims.append(float(cosine_similarity(claim_emb, ev_emb)[0][0]))

    avg = float(np.mean(sims)) if sims else 0.5

    # More decisive thresholds
    if avg > 0.55:
        return {"entail": 0.35, "contra": 0.05, "neutral": 0.60, "used": -1}
    elif avg < 0.30:
        return {"entail": 0.05, "contra": 0.35, "neutral": 0.60, "used": -1}
    else:
        return {"entail": 0.20, "contra": 0.15, "neutral": 0.65, "used": -1}

URL filtering

In [26]:
def is_valid_url(url: str) -> bool:
    bad_ext = (".jpg", ".jpeg", ".png", ".gif", ".webp", ".mp4", ".mp3", ".pdf")
    u = url.lower()
    if any(u.endswith(e) for e in bad_ext) or "image" in u:
        return False
    # Block social media — too noisy for NLI
    domain = u.split("/")[2].replace("www.", "") if "/" in u else ""
    if any(nd in domain for nd in NOISE_DOMAINS):
        return False
    return True

In [27]:
def compute_overlap(query, text):
    q = set(re.findall(r"\w+", query.lower()))
    t = set(re.findall(r"\w+", text.lower()))
    return len(q & t) / len(q) if q else 0.0

Safe search

In [28]:
def safe_google_search(params, retries=3, sleep_sec=2):
    for attempt in range(retries):
        try:    return GoogleSearch(params).get_dict()
        except:
            if attempt < retries - 1:
                time.sleep(sleep_sec * (attempt + 1))
    return {}

Article fetch

In [29]:
import requests
from newspaper import Article

def fetch_article_text(url, min_chars=120, max_chars=2500):
    try:
        head = requests.head(url, timeout=4, headers=HEADERS, allow_redirects=True)
        if head.status_code >= 400:
            return ""
        r = requests.get(url, timeout=8, headers=HEADERS)
        if r.status_code != 200:
            return ""
        a = Article(url)
        a.set_html(r.text)
        a.parse()
        text = " ".join(a.text.split())
        if len(text) < min_chars:
            soup = BeautifulSoup(r.text, "html.parser")
            for tag in soup(["script","style","nav","footer","header"]):
                tag.decompose()
            text = " ".join(soup.get_text().split())
        return text[:max_chars] if len(text) >= min_chars else ""
    except:
        return ""

Retrieve text evidence (CORE)

In [81]:
def retrieve_from_query(query: str, max_results: int = 5) -> list:

    params = {
        "engine":  "google",
        "q":       query,
        "api_key": SERPAPI_KEY,
        "num":     max_results,
    }
    res = safe_google_search(params)
    raw = []

    for r in res.get("organic_results", []):
        url     = r.get("link", "")
        snippet = r.get("snippet", "")
        if not url or not is_valid_url(url):
            continue
        if "reddit.com" in url:
            continue
        if any(str(y) in url for y in ["2014", "2015", "2016", "2017", "2018"]):
            continue

        text = fetch_article_text(url)
        if not text:
            text = snippet

        if text and len(text) >= 40:
            low = text.lower()
            if "login" in low or "subscribe" in low:
                continue
            raw.append({"text": text, "url": url, "is_fact_check": is_fact_checker(url)})

    # Score: 50% keyword overlap + 30% length + 20% domain trust
    scored = []
    for ev in raw:
        overlap      = compute_overlap(query, ev["text"])
        domain_boost = get_domain_boost(ev["url"])
        score        = 0.5 * overlap + 0.3 * (len(ev["text"]) / 2500) + 0.2 * domain_boost
        scored.append((score, ev))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [x[1] for x in scored[10:max_results]]


In [58]:
def retrieve_fact_checks(query: str, max_results: int = 3) -> list:
    STOPWORDS = {"the","a","an","is","are","was","were","of","on","in",
                 "at","to","for","and","or","not","do","does","can","it",
                 "that","this","be","by","from","with","will","has","have"}

    content_words = [
        w for w in re.findall(r"\w+", query.lower())
        if w not in STOPWORDS and len(w) > 2
    ]

    # Scale threshold: 2 words in claim → need 2 matches, 4+ words → need 3
    min_matches = 1

    search_terms = " ".join(content_words[:4])
    fc_sites = (
        "site:snopes.com OR site:politifact.com OR "
        "site:factcheck.org OR site:fullfact.org"
    )
    search_q = f"{search_terms} {fc_sites}"

    params = {
        "engine":  "google",
        "q":       search_q,
        "api_key": SERPAPI_KEY,
        "num":     max_results + 4,
    }
    res = safe_google_search(params)
    raw = []

    for r in res.get("organic_results", []):
        url     = r.get("link", "")
        snippet = r.get("snippet", "")
        title   = r.get("title", "")
        if not url or not is_valid_url(url):
            continue

        combined     = (snippet + " " + title).lower()
        combo_words  = set(re.findall(r"\w+", combined))
        matches      = [w for w in content_words if w in combo_words]

        if len(matches) < min_matches:
            print(f"  [FC SKIP] only {len(matches)} matches (need {min_matches}): {url[:60]}")
            continue

        text = fetch_article_text(url, min_chars=80, max_chars=2000)
        if not text:
            text = snippet   # fall back to snippet — better than nothing

        if text and len(text) >= 40:
            raw.append({
                "text":          text,
                "url":           url,
                "is_fact_check": True,
            })

    return raw[:max_results]

In [54]:
def fetch_article_text(url: str, min_chars: int = 120, max_chars: int = 2500) -> str:
    try:

        head = requests.head(url, timeout=4, headers=HEADERS, allow_redirects=True)
        if head.status_code >= 400:
            return ""

        response = requests.get(url, timeout=8, headers=HEADERS)
        if response.status_code != 200:
            return ""

        a = Article(url)
        a.set_html(response.text)
        a.parse()
        text = " ".join(a.text.split())

        if len(text) < min_chars:
            from bs4 import BeautifulSoup
            soup = BeautifulSoup(response.text, "html.parser")
            for tag in soup(["script","style","nav","footer","header"]):
                tag.decompose()
            text = " ".join(soup.get_text().split())

        return text[:max_chars] if len(text) >= min_chars else ""

    except Exception:
        return ""

In [33]:
def build_search_query(claim: str) -> str:
    #  basic tweet cleaning
    text = re.sub(r'http\S+', '', claim)
    text = re.sub(r'[@#]\w+', '', text)
    text = re.sub(r'^RT\s+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    #  remove incomplete trailing word
    words = text.split()
    if words and len(words[-1]) < 4:
        words = words[:-1]
    text = ' '.join(words)

    # extract content words
    content = [w for w in re.findall(r'\w+', text.lower())
               if w not in STOPWORDS and len(w) > 3]

    # return best 5-6 word query
    if len(content) >= 4:
        return ' '.join(content[:6])
    return text if len(text) > 15 else claim

In [55]:
SCIENCE_DOMAINS = [
    "cdc.gov", "nih.gov", "who.int", "cancer.org", "cancer.gov",
    "pubmed.ncbi.nlm.nih.gov", "mayoclinic.org", "nhs.uk",
    "nasa.gov", "noaa.gov", "usgs.gov",
]

def is_pure_science(query: str) -> bool:
    q = query.lower()
    return any(kw in q for kw in SCIENCE_KEYWORDS)

def retrieve_all_evidence(query: str, use_fact_checks: bool = True) -> list:
    web_ev = retrieve_from_query(query, max_results=6)

    if not use_fact_checks or is_pure_science(query):
        print(f"  [FC SKIP] Pure science claim — using web evidence only")
        return web_ev

    if is_health_fact(query):
        health_sites = "site:cdc.gov OR site:nih.gov OR site:cancer.org OR site:who.int OR site:mayoclinic.org"
        params = {
            "engine":  "google",
            "q":       f"{query} {health_sites}",
            "api_key": SERPAPI_KEY,
            "num":     3,
        }
        res = safe_google_search(params)
        health_ev = []
        for r in res.get("organic_results", []):
            url     = r.get("link", "")
            snippet = r.get("snippet", "")
            if not url or not is_valid_url(url):
                continue
            text = fetch_article_text(url, min_chars=80, max_chars=2000)
            if not text:
                text = snippet
            if text and len(text) >= 40:
                health_ev.append({
                    "text":          text,
                    "url":           url,
                    "is_fact_check": False,
                    "domain_boost":  3.0,
                })

        seen  = set()
        merged = []
        for ev in health_ev + web_ev:
            if ev["url"] not in seen:
                seen.add(ev["url"])
                merged.append(ev)
        return merged

    fc_ev  = retrieve_fact_checks(query, max_results=3)
    seen   = set()
    merged = []
    for ev in fc_ev + web_ev:
        if ev["url"] not in seen:
            seen.add(ev["url"])
            merged.append(ev)
    return merged

In [34]:
def rank_by_overlap_and_trust(claim: str, evidence_list: list) -> list:
    claim_words = set(claim.lower().split())
    scored = []
    for ev in evidence_list:
        ev_words     = set(ev["text"].lower().split())
        overlap      = len(claim_words & ev_words)
        domain_boost = get_domain_boost(ev["url"])
        fc_bonus     = 2.0 if ev.get("is_fact_check") else 0.0
        score        = overlap + domain_boost + fc_bonus
        scored.append((score, ev))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [x[1] for x in scored]


In [35]:
def deduplicate_evidence(evidence_list: list) -> list:
    seen = set()
    out  = []
    for ev in evidence_list:
        key = (ev["url"], ev["text"][:200])
        if key not in seen:
            seen.add(key)
            out.append(ev)
    return out

CLEANING + SENTENCE SELECTION

In [36]:
def select_best_evidence_passages(claim: str, evidence_list: list, top_k: int = 3) -> list:

    claim_words = set(re.findall(r"\w+", claim.lower()))
    scored      = []

    for ev in evidence_list:
        text        = ev["text"]
        ev_words    = set(re.findall(r"\w+", text.lower()))
        kw_score    = len(claim_words & ev_words)
        trust_score = get_domain_boost(ev["url"])
        fc_bonus    = 3.0 if ev.get("is_fact_check") else 0.0
        total_score = kw_score + trust_score + fc_bonus

        # Truncate to 600 chars at a sentence boundary
        truncated = text[:600]
        last_period = truncated.rfind(".")
        if last_period > 300:
            truncated = truncated[:last_period + 1]

        scored.append((total_score, {
            "text":          truncated,
            "url":           ev["url"],
            "is_fact_check": ev.get("is_fact_check", False),
        }))

    scored.sort(key=lambda x: x[0], reverse=True)
    top = [x[1] for x in scored[:top_k]]

    if not top and evidence_list:
        top = [{"text": ev["text"][:600], "url": ev["url"],
                "is_fact_check": ev.get("is_fact_check", False)}
               for ev in evidence_list[:top_k]]
    return top

In [37]:
def clean_text_evidence(evidence_list: list) -> list:
    return [ev for ev in evidence_list
            if len(" ".join(ev["text"].split())) > 50]

In [38]:
def split_sentences(text: str):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s) > 30]

In [ ]:
def select_best_sentences(claim: str, evidence_list, top_k: int = 2):
    claim_words = set(re.findall(r"\w+", claim.lower()))
    scored = []

    for ev in evidence_list:
        text = ev["text"]
        ev_words = set(re.findall(r"\w+", text.lower()))
        score = len(claim_words & ev_words) + get_domain_boost(ev["url"])
        # Keep full passage — NLI needs context, not isolated fragments
        scored.append((score, {"text": text[:500], "url": ev["url"]}))

    scored.sort(key=lambda x: x[0], reverse=True)
    top = [x[1] for x in scored[:top_k]]

    if not top and evidence_list:
        top = [{"text": ev["text"][:500], "url": ev["url"]}
               for ev in evidence_list[:top_k]]
    return top

NLI MODULE

In [39]:
def safe_truncate(text, max_words=200):
    if not text:
        return ""
    return " ".join(text.split()[:max_words])

In [40]:
def fc_article_is_relevant(claim: str, ev: dict, min_shared: int = 2) -> bool:

    claim_words = set(re.findall(r"\w+", claim.lower())) - STOPWORDS
    url_words  = set(re.findall(r"\w+", ev["url"].lower()))
    text_words = set(re.findall(r"\w+", ev["text"][:200].lower()))
    combined   = url_words | text_words

    shared = len(claim_words & combined)
    return shared >= min_shared

In [41]:
def nli_text_score_v2(claim: str, evidence_list: list, top_k: int = 4) -> dict:
    if not evidence_list:
        return {"entail": 0.0, "contra": 0.0, "neutral": 1.0, "used": 0}

    STOPWORDS = {"the","a","an","is","are","was","were","of","on","in",
                 "at","to","for","and","or","not","do","does","can","it",
                 "that","this","be","by","from","with","will","has","have"}
    content_words = set(re.findall(r"\w+", claim.lower())) - STOPWORDS
    min_overlap   = 1 if len(content_words) <= 4 else 2

    e_sum = c_sum = n_sum = weight_sum = 0.0

    for ev in evidence_list[:top_k]:
        text = ev["text"].strip()
        if len(text) < 20:
           continue

        # skip FC articles that are clearly about a different topic
        if ev.get("is_fact_check") and not fc_article_is_relevant(claim, ev):
             print(f"  [FC REJECT] off-topic article: {ev['url'][:60]}")
             continue

        try:
            out = nli({"text": text[:500], "text_pair": claim}, top_k=None)
            if isinstance(out, list) and isinstance(out[0], list):
                out = out[0]
            scores = {}
            for d in out:
                lbl = d["label"].upper()
                if "ENTAIL" in lbl:
                    scores["ENTAILMENT"]    = float(d["score"])
                elif "CONTRADICT" in lbl:
                    scores["CONTRADICTION"] = float(d["score"])
                elif "NEUTRAL" in lbl:
                    scores["NEUTRAL"]       = float(d["score"])
            if len(scores) < 2:
                continue
            w = 3.0 if ev.get("is_fact_check") else 1.0
            MAX_SINGLE = 0.90
            e_sum      += min(scores.get("ENTAILMENT",    0.0), MAX_SINGLE) * w
            c_sum      += min(scores.get("CONTRADICTION", 0.0), MAX_SINGLE) * w
            n_sum      += scores.get("NEUTRAL",       0.0) * w
            weight_sum += w
            print(f"  NLI {'[FC]' if ev.get('is_fact_check') else '    '}: "
                  f"e={scores.get('ENTAILMENT',0):.3f}  "
                  f"c={scores.get('CONTRADICTION',0):.3f}  "
                  f"src={ev['url'][:50]}")
        except Exception as ex:
            print(f"  NLI ERROR: {ex}")

    if weight_sum == 0:
        print("  [FALLBACK] Using embedding similarity")
        return embedding_stance_fallback(claim, evidence_list)

    return {
        "entail":  round(e_sum / weight_sum, 4),
        "contra":  round(c_sum / weight_sum, 4),
        "neutral": round(n_sum / weight_sum, 4),
        "used":    int(weight_sum),
    }

In [42]:
def normalize_nli_scores(scores):
    total = scores["entail"] + scores["contra"] + scores["neutral"]
    if total == 0:
        return scores

    return {
        "entail": scores["entail"] / total,
        "contra": scores["contra"] / total,
        "neutral": scores["neutral"] / total,
        "used": scores["used"]
    }

IMAGE MODULE

In [43]:
def retrieve_image_urls(query, max_results=3):
    params = {
        "engine": "google_images",
        "q": query,
        "api_key": SERPAPI_KEY,
        "num": max_results
    }

    res = GoogleSearch(params).get_dict()

    return [
        r.get("original")
        for r in res.get("images_results", [])
        if r.get("original")
    ]

Load image

In [44]:
def load_image_from_path_or_url(image_input: str, save_dir="/content", timeout=20):
    if image_input is None:
        return None, None, "no_image"
    image_input = str(image_input).strip()
    if image_input.startswith("http://") or image_input.startswith("https://"):
        try:
            r   = requests.get(image_input, timeout=timeout, headers=HEADERS)
            r.raise_for_status()
            img = Image.open(BytesIO(r.content)).convert("RGB")
            img.thumbnail((1024, 1024))
            os.makedirs(save_dir, exist_ok=True)
            local_path = os.path.join(save_dir, "tmp_claim_image.jpg")
            img.save(local_path)
            return img, local_path, "url_ok"
        except Exception as ex:
            return None, None, f"url_failed: {ex}"
    try:
        img = Image.open(image_input).convert("RGB")
        img.thumbnail((1024, 1024))
        return img, image_input, "local_ok"
    except Exception as ex:
        return None, None, f"local_failed: {ex}"


In [45]:
def get_image_for_claim(claim, image_path=None):
    if image_path is not None:
        return image_path

    urls = retrieve_image_urls(claim, max_results=3)
    return urls

In [46]:
def load_image_safe(image_source):
    if image_source is None:
        return None
    try:
        if str(image_source).startswith("http"):
            r = requests.get(image_source, timeout=10, headers=HEADERS)
            return Image.open(BytesIO(r.content)).convert("RGB")
        return Image.open(image_source).convert("RGB")
    except Exception:
        return None


Score images

In [47]:
def score_images_against_claim(claim, image_urls):
    scores = []

    for url in image_urls:
        img = load_image_from_url(url)
        if img is None:
            continue

        sim = compute_clip_score(img, claim)
        scores.append({
            "url": url,
            "score": sim
        })

    scores.sort(key=lambda x: x["score"], reverse=True)
    return scores

OCR BRANCH

In [48]:
def run_ocr(image_path: str) -> str:
    try:
        result = ocr_reader.readtext(image_path, detail=0)
        return " ".join(result).strip()
    except Exception:
        return ""

In [49]:
def ocr_text_claim_alignment_score(claim: str, ocr_text: str) -> float:
    if not ocr_text:
        return 0.0
    c_words = set(re.findall(r"\w+", claim.lower()))
    o_words = set(re.findall(r"\w+", ocr_text.lower()))
    union   = len(c_words | o_words)
    return len(c_words & o_words) / union if union > 0 else 0.0

FUSION + DECISION

In [86]:
def fuse_multimodal_scores(
    text_scores: dict,
    clip_sim:    float,
    ocr_align:   float,
    ocr_text:    str  = "",
    mode:        str  = "text_clip_ocr",
    image_was_loaded: bool = False,   # ADD THIS PARAMETER
) -> dict:
    mode     = mode.lower().strip()
    ocr_text = str(ocr_text) if ocr_text is not None else ""

    # Now uses parameter, not global
    if image_was_loaded and clip_sim != 0.0:
        clip_norm = max(0.0, min(1.0, (clip_sim + 1.0) / 2.0))
    else:
        clip_norm = 0.0

    ocr_norm        = max(0.0, min(1.0, ocr_align))
    entail          = text_scores.get("entail",  0.0)
    contra          = text_scores.get("contra",  0.0)
    neutral         = text_scores.get("neutral", 0.0)
    ocr_has_content = len(ocr_text.strip()) > 10
    mode            = mode.lower().strip()

    if mode == "text_only":
        fused_support = entail
        fused_contra  = contra
        fused_mixed   = neutral + min(entail, contra)
        weights       = {"w_text": 1.0, "w_clip": 0.0, "w_ocr": 0.0}

    elif mode == "text_clip":
        if image_was_loaded and clip_norm > 0:
            w_text, w_clip = 0.85, 0.15
        else:
            w_text, w_clip = 1.0, 0.0
        fused_support = w_text * entail + w_clip * clip_norm * entail
        fused_contra  = w_text * contra + w_clip * (1 - clip_norm) * contra
        fused_mixed   = w_text * (neutral + min(entail, contra))
        weights       = {"w_text": w_text, "w_clip": w_clip, "w_ocr": 0.0}

    else:  # text_clip_ocr
        if image_was_loaded and clip_norm > 0 and ocr_has_content:
            w_text, w_clip, w_ocr = 0.70, 0.15, 0.15
        elif image_was_loaded and clip_norm > 0:
            w_text, w_clip, w_ocr = 0.85, 0.15, 0.0
        else:
            w_text, w_clip, w_ocr = 1.0, 0.0, 0.0
            ocr_norm = 0.0

        fused_support = w_text * entail + w_clip * clip_norm * entail + w_ocr * ocr_norm * entail
        fused_contra  = w_text * contra + w_clip * (1 - clip_norm) * contra + w_ocr * (1 - ocr_norm) * contra
        fused_mixed   = w_text * (neutral + min(entail, contra))
        weights       = {"w_text": w_text, "w_clip": w_clip, "w_ocr": w_ocr}

    return {
        "fused_support": float(fused_support),
        "fused_contra":  float(fused_contra),
        "fused_mixed":   float(fused_mixed),
        "clip_norm":     float(clip_norm),
        "ocr_norm":      float(ocr_norm),
        "ocr_used":      ocr_has_content,
        "weights":       weights,
    }

In [87]:
def final_label_v2(fused, binary_dataset=True):
    support = fused["fused_support"]
    contra  = fused["fused_contra"]

    if binary_dataset:
        MIN_SIGNAL = 0.05
        if support < MIN_SIGNAL and contra < MIN_SIGNAL:
            return "FAKE"
        if support > contra * 1.4 and support >= 0.08:
            return "REAL"
        return "FAKE"

    if contra >= 0.22 and (contra - support) >= 0.05: return "FAKE"
    if support >= 0.18 and (support - contra) >= 0.05: return "REAL"
    return "MISLEADING"

In [51]:
def final_multimodal_label(
    fused:      dict,
    support_th: float = 0.30,
    contra_th:  float = 0.35,
    margin:     float = 0.08,
) -> str:
    support = fused["fused_support"]
    contra  = fused["fused_contra"]
    both_present = support > 0.12 and contra > 0.12
    if both_present and abs(support - contra) < margin * 1.5:
        return "MISLEADING"

    if contra >= contra_th and (contra - support) >= margin:
        return "FAKE"

    if support >= support_th and (support - contra) >= margin:
        return "REAL"

    return "MISLEADING"


EXPLANATION

In [52]:
def generate_explanation(claim, evidence_list, fused, label, text_scores):
    """
    Human-readable reasoning generator for thesis-quality output.
    """

    entail = text_scores.get("entail", 0.0)
    contra = text_scores.get("contra", 0.0)
    neutral = text_scores.get("neutral", 0.0)

    # -----------------------------
    # 1. Evidence summary
    # -----------------------------
    supporting = []
    contradicting = []
    neutral_ev = []

    for ev in evidence_list[:5]:
        text = ev["text"][:200]
        url = ev.get("url", "")

        # simple heuristic (you can later replace with NLI per evidence)
        if "nasa" in text.lower() or "confirms" in text.lower():
            supporting.append(f"{text} ({url})")
        elif "false" in text.lower() or "debunk" in text.lower() or "no evidence" in text.lower():
            contradicting.append(f"{text} ({url})")
        else:
            neutral_ev.append(f"{text} ({url})")

    # -----------------------------
    # 2. Build reasoning text
    # -----------------------------
    explanation = []

    explanation.append(f"Claim Analysis: {claim}\n")

    # Label reasoning
    if label == "REAL":
        explanation.append(
            "Verdict: REAL\n"
            "The evidence overall supports the claim across multiple reliable sources."
        )

    elif label == "FAKE":
        explanation.append(
            "Verdict: FAKE\n"
            "The evidence strongly contradicts the claim across multiple sources."
        )

    else:
        explanation.append(
            "Verdict: MISLEADING\n"
            "The evidence is mixed, with both supporting and contradicting signals."
        )

    # -----------------------------
    # 3. Evidence breakdown
    # -----------------------------
    explanation.append("\nEvidence Breakdown:")

    if supporting:
        explanation.append("\nSupporting Evidence:")
        for s in supporting[:3]:
            explanation.append(f"- {s}")

    if contradicting:
        explanation.append("\nContradicting Evidence:")
        for c in contradicting[:3]:
            explanation.append(f"- {c}")

    if neutral_ev:
        explanation.append("\nNeutral / Irrelevant Evidence:")
        for n in neutral_ev[:2]:
            explanation.append(f"- {n}")

    # -----------------------------
    # 4. Score interpretation
    # -----------------------------
    explanation.append("\nModel Scores:")
    explanation.append(
        f"- Entailment: {entail:.3f}\n"
        f"- Contradiction: {contra:.3f}\n"
        f"- Neutral: {neutral:.3f}"
    )

    # -----------------------------
    # 5. Final interpretation
    # -----------------------------
    explanation.append("\nFinal Interpretation:")

    if contra > entail:
        explanation.append(
            "The contradiction signal is stronger than support, "
            "leading to a FAKE classification."
        )
    elif entail > contra:
        explanation.append(
            "The support signal is stronger than contradiction, "
            "leading to a REAL classification."
        )
    else:
        explanation.append(
            "Signals are balanced, leading to a MISLEADING classification."
        )

    return "\n".join(explanation)

images from URLs

In [ ]:
import requests
from PIL import Image
from io import BytesIO

def download_image_from_url(image_url, save_path="/content/query_image.jpg"):
    r = requests.get(image_url, timeout=20)
    r.raise_for_status()
    img = Image.open(BytesIO(r.content)).convert("RGB")
    img.save(save_path)
    return save_path

In [ ]:
def normalize_claim(claim):
    if isinstance(claim, str):
        return claim
    if isinstance(claim, list):
        if not claim:
            return ""
        if isinstance(claim[0], dict):
            return claim[0].get("claim") or claim[0].get("text", "")
        return " ".join(str(x) for x in claim)
    if isinstance(claim, dict):
        return claim.get("claim") or claim.get("text") or str(claim)
    return str(claim)

MAIN MULTIMODAL PIPELINE

In [60]:
def main_pipeline_v2(
    claim:          str,
    image_path:     str  = None,
    top_k_text:     int  = 4,
    mode:           str  = "text_clip_ocr",
    use_fact_checks:bool = True,
    binary_dataset: bool = True,
) -> dict:

    # Fix 3: clean tweet noise before searching
    search_query = clean_tweet_claim(claim)

    # Text branch
    retrieved = retrieve_all_evidence(search_query, use_fact_checks=use_fact_checks)
    deduped   = deduplicate_evidence(retrieved)
    cleaned   = clean_text_evidence(deduped)
    ranked    = rank_by_overlap_and_trust(search_query, cleaned)
    top_text  = select_best_evidence_passages(search_query, ranked, top_k=top_k_text)

    if not top_text:
        return {"claim": claim, "label": "FAKE" if binary_dataset else "NO_EVIDENCE",
                "confidence": 0.0, "explanation": "No evidence retrieved.",
                "text_scores": {"entail":0.0,"contra":0.0,"neutral":1.0,"used":0},
                "fused": {"fused_support":0.0,"fused_contra":0.0,"fused_mixed":0.0,
                          "clip_norm":0.0,"ocr_norm":0.0,"ocr_used":False,"weights":{}},
                "evidence_text": []}

    # Fix 2: DeBERTa NLI + Fix 4: embedding fallback
    text_scores = nli_text_score_v2(claim, top_text, top_k=top_k_text)

    # Image branch (unchanged)
    clip_sim = 0.0
    ocr_text = "" # Initialize ocr_text as a string
    ocr_align = 0.0
    image_used = False
    if image_path:
        img_temp, local_path, _ = load_image_from_path_or_url(image_path)
        if img_temp:
            image_used = True
            clip_sim   = compute_clip_score(img_temp, claim)
            try:
                ocr_text  = run_ocr(local_path) if local_path else ""
                ocr_align = ocr_text_claim_alignment_score(claim, ocr_text)
            except Exception:
                ocr_text = ""

    fused = fuse_multimodal_scores(
        text_scores=text_scores, clip_sim=clip_sim,
        ocr_align=ocr_align, ocr_text=ocr_text, mode=mode,
    )

    # Fix 1: binary-aware label
    label      = final_label_v2(fused, binary_dataset=binary_dataset)
    confidence = round(max(fused["fused_support"], fused["fused_contra"]) * 100, 2)
    explanation = generate_explanation(
                          claim,
                          top_text,
                          fused,
                          label,
                          text_scores
                         )
    return {
        "claim": claim, "label": label, "confidence": confidence,
        "text_scores": text_scores, "fused": fused,
        "evidence_text": top_text, "explanation": explanation,
        "image_used": image_used, "search_query": search_query,
    }

TEST RUN

In [ ]:
out1 = main_pipeline_v2("NASA confirms water on Mars",
       image_path="https://mars.nasa.gov/system/resources/detail_files/25629_PIA23764-1041.jpg")
out2 = main_pipeline_v2("The Earth is flat")
out3 = main_pipeline_v2("Smoking causes lung cancer")

print(f"Test 1 (expect MISLEADING/REAL): {out1['label']}  conf={out1['confidence']}")
print(f"Test 2 (expect FAKE):            {out2['label']}  e={out2['text_scores']['entail']:.3f}  c={out2['text_scores']['contra']:.3f}")
print(f"Test 3 (expect REAL):            {out3['label']}  e={out3['text_scores']['entail']:.3f}  c={out3['text_scores']['contra']:.3f}")

  [FC SKIP] only 1 matches (need 2): https://www.snopes.com/news/2020/12/08/galactic-federation-n
  [FC SKIP] only 1 matches (need 2): https://www.snopes.com/news/2022/09/21/mars-is-littered-with
  [FC SKIP] only 0 matches (need 2): https://www.snopes.com/
  [FC SKIP] only 1 matches (need 2): https://www.politifact.com/factchecks/2014/mar/12/rush-limba
  NLI [FC]: e=0.001  c=0.001  src=https://www.politifact.com/factchecks/2024/feb/23/
  [FC REJECT] off-topic article: https://www.snopes.com/fact-check/17-year-old-astronaut-trai
  NLI     : e=0.151  c=0.002  src=https://www.nasa.gov/news-release/nasa-confirms-ev
  NLI     : e=0.921  c=0.006  src=https://www.reuters.com/science/nasa-rover-detects
  [FC SKIP] only 0 matches (need 1): https://www.factcheck.org/2015/03/cruz-on-the-global-cooling
  NLI [FC]: e=0.002  c=0.926  src=https://fullfact.org/online/earth-is-spherical-not
  NLI [FC]: e=0.001  c=0.969  src=https://www.politifact.com/factchecks/2024/may/29/
  [FC REJECT] off-topic arti

In [ ]:
out = main_pipeline_v2(
    "NASA confirms water on Mars",
    image_path="https://mars.nasa.gov/system/resources/detail_files/25629_PIA23764-1041.jpg"
)
print(f"\nTest 1 (expect MISLEADING): {out['label']}  confidence={out['confidence']}")
print(f"  entail={out['text_scores']['entail']:.3f}  "
      f"contra={out['text_scores']['contra']:.3f}  "
      f"used={out['text_scores']['used']}")
print(f"  fused_support={out['fused']['fused_support']:.3f}  "
      f"fused_contra={out['fused']['fused_contra']:.3f}")
print("  Sources:")
for ev in out["evidence_text"]:
    fc = " [FC]" if ev.get("is_fact_check") else ""
    print(f"    {fc} {ev['url'][:70]}")

out2 = main_pipeline_v2("The Earth is flat")
print(f"\nTest 2 (expect FAKE):       {out2['label']}  "
      f"entail={out2['text_scores']['entail']:.3f}  "
      f"contra={out2['text_scores']['contra']:.3f}")

out3 = main_pipeline_v2("Smoking causes lung cancer")
print(f"\nTest 3 (expect REAL):       {out3['label']}  "
      f"entail={out3['text_scores']['entail']:.3f}  "
      f"contra={out3['text_scores']['contra']:.3f}")

  [FC SKIP] only 0 matches (need 2): https://www.snopes.com/
  [FC SKIP] only 1 matches (need 2): https://www.snopes.com/news/2020/12/08/galactic-federation-n
  [FC SKIP] only 1 matches (need 2): https://www.snopes.com/news/2022/09/21/mars-is-littered-with
  [FC SKIP] only 1 matches (need 2): https://www.politifact.com/factchecks/2014/mar/12/rush-limba
  NLI [FC]: e=0.001  c=0.001  src=https://www.politifact.com/factchecks/2024/feb/23/
  [FC REJECT] off-topic article: https://www.snopes.com/fact-check/17-year-old-astronaut-trai
  NLI     : e=0.150  c=0.002  src=https://www.nasa.gov/news-release/nasa-confirms-ev
  NLI     : e=0.013  c=0.003  src=https://www.nasa.gov/missions/mars-science-laborat

Test 1 (expect MISLEADING): REAL  confidence=15.6
  entail=0.041  contra=0.002  used=4
  fused_support=0.156  fused_contra=0.001
  Sources:
     [FC] https://www.politifact.com/factchecks/2024/feb/23/instagram-posts/nasa
     [FC] https://www.snopes.com/fact-check/17-year-old-astronaut-training

DuckDuckGo Search

In [90]:
!pip install -q duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 61.6 MB/s eta 0:00:00


In [98]:
!pip uninstall -y duckduckgo-search
!pip install -q ddgs

Found existing installation: duckduckgo_search 8.1.1
Uninstalling duckduckgo_search-8.1.1:
  Successfully uninstalled duckduckgo_search-8.1.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 3.0 MB/s eta 0:00:00


In [97]:
import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("ddgs").setLevel(logging.ERROR)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [99]:
from ddgs import DDGS  # changed from duckduckgo_search

In [104]:
import time

def retrieve_from_query_ddg(query: str, max_results: int = 3) -> list:
    raw = []
    try:
        time.sleep(1)  # ADD THIS — prevents rate limiting
        with DDGS() as ddgs:
            hits = list(ddgs.text(query, max_results=max_results + 3))

        for r in hits:
            url     = r.get("href", "")
            snippet = r.get("body", "")
            if not url or not is_valid_url(url):
                continue
            if "reddit.com" in url:
                continue
            if any(str(y) in url for y in ["2014", "2015", "2016", "2017", "2018"]):
                continue

            text = fetch_article_text(url)
            if not text:
                text = snippet

            if text and len(text) >= 40:
                low = text.lower()
                if "login" in low or "subscribe" in low:
                    continue
                raw.append({"text": text, "url": url, "is_fact_check": is_fact_checker(url)})

    except Exception as e:
        print(f"  DDG ERROR: {e}")

    # Same scoring as original: 50% overlap + 30% length + 20% domain trust
    scored = []
    for ev in raw:
        overlap      = compute_overlap(query, ev["text"])
        domain_boost = get_domain_boost(ev["url"])
        score        = 0.5 * overlap + 0.3 * (len(ev["text"]) / 2500) + 0.2 * domain_boost
        scored.append((score, ev))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [x[1] for x in scored[:max_results]]  # fixed slice bug too


def retrieve_fact_checks_ddg(query: str, max_results: int = 2) -> list:
    STOPWORDS_LOCAL = {"the","a","an","is","are","was","were","of","on","in",
                       "at","to","for","and","or","not","do","does","can","it",
                       "that","this","be","by","from","with","will","has","have"}

    content_words = [
        w for w in re.findall(r"\w+", query.lower())
        if w not in STOPWORDS_LOCAL and len(w) > 2
    ]
    min_matches  = 1
    search_terms = " ".join(content_words[:4])
    fc_query     = (
        f"{search_terms} site:snopes.com OR site:politifact.com OR "
        f"site:factcheck.org OR site:fullfact.org"
    )

    raw = []
    try:
        with DDGS() as ddgs:
            hits = list(ddgs.text(fc_query, max_results=max_results + 4))

        for r in hits:
            url     = r.get("href", "")
            snippet = r.get("body", "")
            title   = r.get("title", "")
            if not url or not is_valid_url(url):
                continue

            # Same relevance check as original
            combined    = (snippet + " " + title).lower()
            combo_words = set(re.findall(r"\w+", combined))
            matches     = [w for w in content_words if w in combo_words]

            if len(matches) < min_matches:
                print(f"  [FC SKIP] only {len(matches)} matches (need {min_matches}): {url[:60]}")
                continue

            text = fetch_article_text(url, min_chars=80, max_chars=2000)
            if not text:
                text = snippet

            if text and len(text) >= 40:
                raw.append({
                    "text":          text,
                    "url":           url,
                    "is_fact_check": True,
                })

    except Exception as e:
        print(f"  DDG FC ERROR: {e}")

    return raw[:max_results]


def retrieve_all_evidence_ddg(query: str, use_fact_checks: bool = True) -> list:
    web_ev = retrieve_from_query_ddg(query, max_results=3)

    if not use_fact_checks or is_pure_science(query):
        print(f"  [FC SKIP] Pure science claim — using web evidence only")
        return web_ev

    if is_health_fact(query):
        # Health claims — search trusted health domains
        health_query = f"{query} site:cdc.gov OR site:nih.gov OR site:who.int OR site:mayoclinic.org"
        health_ev = []
        try:
            with DDGS() as ddgs:
                hits = list(ddgs.text(health_query, max_results=3))
            for r in hits:
                url     = r.get("href", "")
                snippet = r.get("body", "")
                if not url or not is_valid_url(url):
                    continue
                text = fetch_article_text(url, min_chars=80, max_chars=2000)
                if not text:
                    text = snippet
                if text and len(text) >= 40:
                    health_ev.append({
                        "text":          text,
                        "url":           url,
                        "is_fact_check": False,
                        "domain_boost":  3.0,
                    })
        except Exception as e:
            print(f"  DDG HEALTH ERROR: {e}")

        seen   = set()
        merged = []
        for ev in health_ev + web_ev:
            if ev["url"] not in seen:
                seen.add(ev["url"])
                merged.append(ev)
        return merged

    # General claims — use fact checkers
    fc_ev  = retrieve_fact_checks_ddg(query, max_results=2)
    seen   = set()
    merged = []
    for ev in fc_ev + web_ev:
        if ev["url"] not in seen:
            seen.add(ev["url"])
            merged.append(ev)
    return merged

In [108]:
def final_label_calibrated(fused, text_scores, evidence_list):
    support = fused["fused_support"]
    contra  = fused["fused_contra"]
    entail  = text_scores.get("entail", 0.0)
    contra_raw = text_scores.get("contra", 0.0)

    # Count how many sources have high entailment
    high_entail_count = sum(
        1 for ev in evidence_list
        if ev.get("text") and entail > 0.3
    )

    # Count trusted domains in evidence
    trusted_count = sum(
        1 for ev in evidence_list
        if get_domain_boost(ev.get("url", "")) >= 2.0
    )

    MIN_SIGNAL = 0.05

    # Strong contradiction → FAKE
    if contra >= 0.25 and (contra - support) >= 0.08:
        return "FAKE"

    # Strong support → REAL
    if support >= 0.20 and (support - contra) >= 0.08:
        return "REAL"

    # Weak signals but trusted sources lean toward entailment → REAL
    if trusted_count >= 2 and entail > contra_raw:
        return "REAL"

    # No evidence at all → FAKE
    if support < MIN_SIGNAL and contra < MIN_SIGNAL:
        return "FAKE"

    # Default: lean FAKE only if contra has any signal
    if contra_raw > 0.05:
        return "FAKE"

    return "REAL"  # changed from FAKE — benefit of doubt with no contra signal

In [106]:
import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("ddgs").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)
import shutil

modes       = ["text_only", "text_clip", "text_clip_ocr"]
summaries   = []
all_outputs = {}

for m in modes:
    df_mode, summary, cm, report = evaluate_mode_v2(
        dataset=final_eval_dataset,
        mode=m,
        top_k_text=4,
        use_fc=True,
        per_sample_timeout=90,
        verbose=True,
    )
    path = f"/content/ablation_v2_{m}.csv"
    df_mode.to_csv(path, index=False)
    print(f"Saved: {path}")
    all_outputs[m] = {"df": df_mode, "summary": summary}
    summaries.append(summary)

summary_df = (
    pd.DataFrame(summaries)
    .sort_values("f1_macro", ascending=False)
    .reset_index(drop=True)
)
print("\n" + "="*60)
print("ABLATION SUMMARY")
print(summary_df[["mode","n_valid","accuracy","precision_macro",
                   "recall_macro","f1_macro"]].to_string(index=False))
summary_df.to_csv("/content/ablation_v2_summary.csv", index=False)

# Save to Drive
for m in modes:
    try:
        shutil.copy(f"/content/ablation_v2_{m}.csv",
                    f"/content/drive/MyDrive/ablation_v2_{m}.csv")
    except: pass
try:
    shutil.copy("/content/ablation_v2_summary.csv",
                "/content/drive/MyDrive/ablation_v2_summary.csv")
    print("All saved to Drive")
except: pass

Running text_only:   0%|          | 0/99 [00:00<?, ?it/s]

  [FC REJECT] off-topic article: https://www.snopes.com/fact-check/ai-photo-missiles-tel-aviv
  NLI     : e=0.010  c=0.001  src=https://www.pbs.org/newshour/world/hezbollah-fires
  NLI [FC]: e=0.315  c=0.059  src=https://api.politifact.com/article/2023/oct/19/did


Running text_only:   1%|          | 1/99 [00:13<22:14, 13.62s/it]

  NLI     : e=0.011  c=0.906  src=https://www.timesofisrael.com/deadly-duel-photos-c
  [FC SKIP] only 0 matches (need 1): https://www.politifact.com/
  [FC SKIP] only 0 matches (need 1): https://www.factcheck.org/
  [FC SKIP] only 0 matches (need 1): https://www.politifact.com/factchecks/
  [FC SKIP] only 0 matches (need 1): https://www.factcheck.org/fake-news/


Running text_only:   2%|▏         | 2/99 [00:24<19:05, 11.81s/it]

...

  DDG FC ERROR: ('error sending request for url (https://html.duckduckgo.com/html/)', 'https://html.duckduckgo.com/html/')
  NLI     : e=0.001  c=0.000  src=https://www.npr.org/2026/04/03/nx-s1-5772611/blake
  NLI     : e=0.002  c=0.000  src=https://www.latimes.com/entertainment-arts/movies/
  NLI     : e=0.009  c=0.000  src=https://variety.com/2026/film/news/blake-lively-la


Running text_clip_ocr:  99%|█████████▉| 98/99 [25:08<00:15, 15.27s/it]

  DDG FC ERROR: No results found.
  NLI     : e=0.002  c=0.990  src=https://www.aol.com/madison-everything-know-michel
  NLI     : e=0.699  c=0.005  src=https://variety.com/2024/tv/news/michelle-pfeiffer
  NLI     : e=0.172  c=0.007  src=https://deadline.com/2024/08/michelle-pfeiffer-con


Running text_clip_ocr: 100%|██████████| 99/99 [25:26<00:00, 15.42s/it]

  DDG FC ERROR: ('error sending request for url (https://html.duckduckgo.com/html/)', 'https://html.duckduckgo.com/html/')
  NLI     : e=0.997  c=0.001  src=https://www.themirror.com/entertainment/jay-zs-rap
  NLI     : e=0.497  c=0.025  src=https://www.foxnews.com/entertainment/jay-zs-sexua
  NLI     : e=0.751  c=0.015  src=https://www.tmz.com/2024/12/13/jay-z-diddy-rape-ac
MODE: text_clip_ocr
  mode                 text_clip_ocr
  n_total              99
  n_valid              99
  n_error              0
  accuracy             0.7172
  precision_macro      0.7425
  recall_macro         0.7188
  f1_macro             0.7105

Prediction distribution:
pred
FAKE    65
REAL    34
Name: count, dtype: int64

              precision    recall  f1-score   support

        FAKE       0.66      0.88      0.75        49
        REAL       0.82      0.56      0.67        50

    accuracy                           0.72        99
   macro avg       0.74      0.72      0.71        99
weighted avg    

In [107]:
import numpy as np
from sklearn.metrics import accuracy_score
import pandas as pd

df_valid = pd.read_csv("/content/drive/MyDrive/ablation_v2_text_clip_ocr.csv")
df_valid = df_valid[df_valid["pred"] != "ERROR"].copy()

best_acc, best_ratio = 0, 1.4

for ratio in np.arange(0.8, 2.5, 0.1):
    def predict_with_ratio(row, r):
        s, c = row["fused_support"], row["fused_contra"]
        if s < 0.05 and c < 0.05: return "FAKE"
        if s > c * r and s >= 0.08: return "REAL"
        return "FAKE"
    df_valid["pred_tuned"] = df_valid.apply(lambda r: predict_with_ratio(r, ratio), axis=1)
    acc = accuracy_score(df_valid["true"], df_valid["pred_tuned"])
    if acc > best_acc:
        best_acc, best_ratio = acc, ratio

print(f"Best ratio: {best_ratio:.1f}")
print(f"Tuned accuracy: {best_acc:.4f}")
print(f"Current accuracy: 0.7172")
print(f"Improvement: +{round(best_acc - 0.7172, 4)}")

Best ratio: 0.8
Tuned accuracy: 0.7172
Current accuracy: 0.7172
Improvement: +-0.0


In [110]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("/content/drive/MyDrive/ablation_v2_text_clip_ocr.csv")
df = df[df["pred"] != "ERROR"].copy()

def predict_calibrated(row):
    support = row["fused_support"]
    contra  = row["fused_contra"]
    entail  = row["text_entail"]
    contra_raw = row["text_contra"]

    if contra >= 0.25 and (contra - support) >= 0.08:
        return "FAKE"
    if support >= 0.20 and (support - contra) >= 0.08:
        return "REAL"
    if support < 0.05 and contra < 0.05:
        if contra_raw > 0.05:
            return "FAKE"
        return "REAL"  # no signal, no contra → benefit of doubt
    if contra_raw > 0.05:
        return "FAKE"
    return "REAL"

df["pred_calibrated"] = df.apply(predict_calibrated, axis=1)

acc = accuracy_score(df["true"], df["pred_calibrated"])
print(f"Calibrated accuracy: {acc:.4f}  (was 0.7172)")
print(classification_report(df["true"], df["pred_calibrated"], labels=["FAKE","REAL"]))

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(df["true"], df["pred_calibrated"], labels=["FAKE","REAL"])
print(pd.DataFrame(cm, index=["FAKE","REAL"], columns=["FAKE","REAL"]))

Calibrated accuracy: 0.6364  (was 0.7172)
              precision    recall  f1-score   support

        FAKE       0.74      0.41      0.53        49
        REAL       0.60      0.86      0.70        50

    accuracy                           0.64        99
   macro avg       0.67      0.63      0.62        99
weighted avg       0.67      0.64      0.62        99

      FAKE  REAL
FAKE    20    29
REAL     7    43


### V2 EVALUATION LOOP WITH ABLATIONS

In [61]:
import os
import time
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)

LABEL_ORDER = ["FAKE", "MISLEADING", "REAL"]

In [62]:
import threading

def run_with_timeout(fn, args=(), kwargs={}, timeout_sec=30, fallback=None):
    result = [fallback]
    exc = [None]

    def target():
        try:
            result[0] = fn(*args, **kwargs)
        except Exception as e:
            exc[0] = e

    t = threading.Thread(target=target, daemon=True)
    t.start()
    t.join(timeout_sec)

    if t.is_alive():
        print(f"  Per-sample timeout ({timeout_sec}s)")
        return fallback, True

    if exc[0] is not None:
        raise exc[0]

    return result[0], False

MAIN SAMPLE RUNNER

In [63]:
def run_one_sample(item, mode="text_clip_ocr", top_k_text=4, use_fc=True):
    claim    = item["claim"]
    img_path = item.get("image_path", None)

    # TEXT branch
    try:
        search_query = clean_tweet_claim(claim)
        retrieved    = retrieve_all_evidence(search_query, use_fact_checks=use_fc)
        cleaned      = clean_text_evidence(retrieved)
        ranked       = rank_by_overlap_and_trust(search_query, cleaned)
        top_text     = select_best_evidence_passages(search_query, ranked, top_k=top_k_text)
        text_scores  = nli_text_score(claim, top_text, top_k=top_k_text)
    except Exception as e:
        print(f"  TEXT ERROR: {e}")
        top_text    = []
        text_scores = {"entail": 0.0, "contra": 0.0, "neutral": 1.0, "used": 0}

    # IMAGE branch
    clip_sim        = 0.0
    ocr_text        = ""
    ocr_align       = 0.0
    image_was_loaded = False

    if img_path and mode in {"text_clip", "text_clip_ocr"}:
        try:
            img_temp, local_path, status = load_image_from_path_or_url(img_path)
            if img_temp is not None:
                image_was_loaded = True
                clip_sim         = compute_clip_score(img_temp, claim)
                if mode == "text_clip_ocr" and local_path:
                    try:
                        ocr_text  = run_ocr(local_path)
                        ocr_align = ocr_text_claim_alignment_score(claim, ocr_text)
                    except:
                        ocr_text = ""
        except Exception as e:
            print(f"  IMAGE ERROR: {e}")

    # FUSION — now correctly knows if image was loaded
    fused = fuse_multimodal_scores(
        text_scores     = text_scores,
        clip_sim        = clip_sim,
        ocr_align       = ocr_align,
        ocr_text        = ocr_text,
        mode            = mode,
        image_was_loaded = image_was_loaded,
    )

    pred       = final_label(fused, binary_dataset=True)
    confidence = round(max(fused["fused_support"], fused["fused_contra"]) * 100, 2)

    return {
        "claim":          claim,
        "true":           item["label"],
        "pred":           pred,
        "mode":           mode,
        "confidence":     confidence,
        "image_was_loaded": image_was_loaded,
        "clip_sim":       clip_sim,
        "text_entail":    text_scores["entail"],
        "text_contra":    text_scores["contra"],
        "fused_support":  fused["fused_support"],
        "fused_contra":   fused["fused_contra"],
        "weights":        fused["weights"],
    }

In [109]:
def run_one_sample_ablation(
    item:        dict,
    mode:        str  = "text_clip_ocr",
    top_k_text:  int  = 4,          # changed from 2 → 4
    use_fc:      bool = True,
    binary_dataset: bool = True,    # XFactA is binary
) -> dict:

    claim      = item["claim"]
    true_label = item["label"]
    img_path   = item.get("image_path", None)

    # TEXT BRANCH — now uses clean_tweet_claim + v2 NLI
    try:
        search_query = clean_tweet_claim(claim)
        #retrieved    = retrieve_all_evidence(search_query, use_fact_checks=use_fc)
        retrieved = retrieve_all_evidence_ddg(search_query, use_fact_checks=use_fc)
        cleaned      = clean_text_evidence(retrieved)
        ranked       = rank_by_overlap_and_trust(search_query, cleaned)
        top_text     = select_best_evidence_passages(search_query, ranked, top_k=top_k_text)
        text_scores  = nli_text_score_v2(claim, top_text, top_k=top_k_text)
    except Exception as e:
        print(f"  TEXT ERROR [{claim[:40]}]: {e}")
        top_text    = []
        text_scores = {"entail": 0.0, "contra": 0.0, "neutral": 1.0, "used": 0}

    # IMAGE BRANCH — unchanged
    clip_sim = 0.0; ocr_text = ""; ocr_align = 0.0
    image_used = False; actual_image_source = None; ocr_local_path = None
    image_status = "no_image_input"

    use_clip = mode in {"text_clip", "text_clip_ocr"}
    use_ocr  = mode == "text_clip_ocr"

    if img_path and (use_clip or use_ocr):
        try:
            img_temp, local_path, status = load_image_from_path_or_url(img_path)
            image_status = status
            if img_temp is not None:
                image_used = True; actual_image_source = img_path
                ocr_local_path = local_path
                if use_clip:
                    clip_sim = compute_clip_score(img_temp, claim)
                if use_ocr and ocr_local_path:
                    try:    ocr_text = run_ocr(ocr_local_path)
                    except: ocr_text = ""
                    ocr_align = ocr_text_claim_alignment_score(claim, ocr_text)
        except Exception as e:
            print(f"  IMAGE ERROR [{claim[:40]}]: {e}")

    # FUSION
    fused = fuse_multimodal_scores(
        text_scores=text_scores, clip_sim=clip_sim,
        ocr_align=ocr_align, ocr_text=ocr_text, mode=mode,
    )

    # v2 label — binary-aware, never wastes a prediction on MISLEADING
    #pred       = final_label_v2(fused, binary_dataset=binary_dataset)
    pred = final_label_calibrated(fused, text_scores, top_text)
    confidence = round(max(fused["fused_support"], fused["fused_contra"]) * 100, 2)

    try:
        explanation = generate_multimodal_explanation(
            claim, top_text, clip_sim, ocr_text, fused, pred
        )
    except Exception:
        explanation = "Explanation unavailable"

    return {
        "claim":          claim,
        "true":           true_label,
        "pred":           pred,
        "mode":           mode,
        "confidence":     confidence,
        "evidence_texts": [ev["text"] for ev in top_text],
        "evidence_urls":  [ev["url"]  for ev in top_text],
        "fc_sources":     [ev["url"]  for ev in top_text if ev.get("is_fact_check")],
        "explanation":    explanation,
        "text_entail":    text_scores["entail"],
        "text_contra":    text_scores["contra"],
        "text_neutral":   text_scores["neutral"],
        "clip_sim":       clip_sim,
        "ocr_text_len":   len(ocr_text),
        "ocr_used":       fused.get("ocr_used", False),
        "fused_support":  fused["fused_support"],
        "fused_contra":   fused["fused_contra"],
        "fused_mixed":    fused.get("fused_mixed", 0.0),
        "image_used":     image_used,
        "image_source":   actual_image_source,
        "image_status":   image_status,
        "evidence_count": len(top_text),
        "clip_norm":      fused["clip_norm"],
        "ocr_norm":       fused["ocr_norm"],
        "weights":        fused["weights"],
        "search_query":   search_query if 'search_query' in dir() else claim,
    }

RUN ALL ABLATIONS + COMPARE

In [65]:
def run_ablation_suite(dataset, top_k_text=3, per_sample_timeout=60, use_fc=True):
    modes       = ["text_only", "text_clip", "text_clip_ocr"]
    all_outputs = {}
    summaries   = []

    for m in modes:
        df_mode, summary, cm, report = evaluate_mode(
            dataset=dataset,
            mode=m,
            top_k_text=top_k_text,
            use_fc=use_fc,
            per_sample_timeout=per_sample_timeout,
            verbose=True,
        )
        out_path = f"/content/ablation_{m}_full_outputs.csv"
        df_mode.to_csv(out_path, index=False)
        print(f"  Saved: {out_path}")
        all_outputs[m] = {"df": df_mode, "summary": summary}
        summaries.append(summary)

    summary_df = (
        pd.DataFrame(summaries)
        .sort_values("f1_macro", ascending=False)
        .reset_index(drop=True)
    )
    print("\n" + "#" * 70)
    print("ABLATION SUMMARY (sorted by macro-F1)")
    print(summary_df[["mode","n_valid","accuracy","precision_macro","recall_macro","f1_macro"]].to_string(index=False))
    print("#" * 70)
    summary_df.to_csv("/content/ablation_summary.csv", index=False)
    return all_outputs, summary_df


check raw NLI scores

In [ ]:
def debug_claim(claim, true_label):
    print(f"\n{'='*60}")
    print(f"Claim: {claim!r}  (true={true_label})")
    retrieved = retrieve_all_evidence(claim, use_fact_checks=True)
    cleaned   = clean_text_evidence(retrieved)
    ranked    = rank_by_overlap_and_trust(claim, cleaned)
    top_text  = select_best_evidence_passages(claim, ranked, top_k=3)

    if not top_text:
        print("  NO EVIDENCE retrieved")
        return

    for i, ev in enumerate(top_text):
        fc_tag = " [FACT-CHECK]" if ev.get("is_fact_check") else ""
        print(f"  Evidence {i+1}{fc_tag}: {ev['text'][:120]}...")
        inp    = {"text": ev["text"], "text_pair": claim}
        out    = nli(inp, top_k=None)
        if isinstance(out[0], list):
            out = out[0]
        scores = {d["label"].upper(): round(d["score"], 3) for d in out}
        print(f"  NLI scores: {scores}")

    ts = nli_text_score(claim, top_text)
    print(f"  Averaged → entail={ts['entail']:.3f}  contra={ts['contra']:.3f}  neutral={ts['neutral']:.3f}")
    print(f"  used={ts['used']}")

    fused = fuse_multimodal_scores(ts, 0.0, 0.0, mode="text_only")
    label = final_multimodal_label(fused)
    print(f"  → Predicted: {label}  (fused_support={fused['fused_support']:.3f}  fused_contra={fused['fused_contra']:.3f}  mixed={fused['fused_mixed']:.3f})")

debug_claim("NASA confirms water on Mars", "MISLEADING")
debug_claim("The Earth is flat",           "FAKE")
debug_claim("Smoking causes lung cancer",  "REAL")


Claim: 'NASA confirms water on Mars'  (true=MISLEADING)
  [FC SKIP] only 0 matches (need 2): https://www.snopes.com/
  [FC SKIP] only 1 matches (need 2): https://www.snopes.com/news/2020/12/08/galactic-federation-n
  [FC SKIP] only 1 matches (need 2): https://www.snopes.com/news/2022/09/21/mars-is-littered-with
  [FC SKIP] only 1 matches (need 2): https://www.politifact.com/factchecks/2014/mar/12/rush-limba
  Evidence 1 [FACT-CHECK]: No, NASA is not editing images from Canada and passing them off as photos of Mars. Similar claims have been debunked num...
  NLI scores: {'NEUTRAL': 0.999, 'CONTRADICTION': 0.0, 'ENTAILMENT': 0.0}
  Evidence 2 [FACT-CHECK]: Claim: A 17-year-old girl named Alyssa Carson is being trained by NASA to become an astronaut. Rating: About this rating...
  NLI scores: {'CONTRADICTION': 0.904, 'NEUTRAL': 0.093, 'ENTAILMENT': 0.003}
  Evidence 3: These dark, narrow, 100 meter-long streaks called recurring slope lineae flowing downhill on Mars are inferred to have b

In [93]:
item = final_eval_dataset[0]
row  = run_one_sample_ablation(
    item, mode="text_only",
    top_k_text=4, use_fc=True, binary_dataset=True,
)
print("Pred:", row["pred"])
print("True:", row["true"])
print("Confidence:", row["confidence"])

/tmp/ipykernel_2349/2161664455.py:6: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use time

  NLI     : e=0.204  c=0.178  src=https://emojis.wiki/high-voltage/
  NLI     : e=0.040  c=0.019  src=https://www.emojiall.com/en/emoji/⚡️
  NLI     : e=0.012  c=0.016  src=https://emojiguide.org/high-voltage
Pred: FAKE
True: FAKE
Confidence: 8.55


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [94]:
import re

def clean_tweet_claim(claim: str) -> str:
    # Remove emojis
    claim = re.sub(r'[^\x00-\x7F]+', ' ', claim)
    # Remove URLs
    claim = re.sub(r'http\S+', '', claim)
    # Remove mentions and hashtags
    claim = re.sub(r'[@#]\w+', '', claim)
    # Remove RT prefix
    claim = re.sub(r'^RT\s+', '', claim)
    # Clean extra spaces
    claim = re.sub(r'\s+', ' ', claim).strip()
    return claim if len(claim) > 15 else claim

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [95]:
# Test clean_tweet_claim first
claim = "⚡️A picture of war in the sky of Israel. A Hezbollah rocket while being approached by an interceptor missile."
print("Cleaned:", clean_tweet_claim(claim))

Cleaned: A picture of war in the sky of Israel. A Hezbollah rocket while being approached by an interceptor missile.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [96]:
item = final_eval_dataset[0]
row  = run_one_sample_ablation(
    item, mode="text_only",
    top_k_text=4, use_fc=True, binary_dataset=True,
)
print("Pred:", row["pred"])
print("True:", row["true"])
print("Confidence:", row["confidence"])

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Pred: FAKE
True: FAKE
Confidence: 0.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


Find the best threshold on your results



In [89]:
import shutil

all_outputs = {}
summaries   = []

df_mode, summary, cm, report = evaluate_mode_v2(
    dataset=final_eval_dataset,
    mode="text_only",      # ONE MODE ONLY
    top_k_text=4,
    use_fc=True,
    per_sample_timeout=90,
    verbose=True,
)
all_outputs["text_only"] = {"df": df_mode, "summary": summary}
summaries.append(summary)

df_mode.to_csv("/content/ablation_v2_text_only.csv", index=False)

# Save to Drive immediately
try:
    shutil.copy("/content/ablation_v2_text_only.csv",
                "/content/drive/MyDrive/ablation_v2_text_only.csv")
    print("Saved to Drive")
except Exception as e:
    print(f"Drive save failed: {e}")

Running text_only:   0%|          | 0/99 [00:00<?, ?it/s]

  [FC SKIP] only 0 matches (need 1): https://www.snopes.com/fact-check/hamas-wedding-girls/
  [FC SKIP] only 0 matches (need 1): https://fullfact.org/conflict/
  NLI [FC]: e=0.285  c=0.532  src=https://www.snopes.com/fact-check/photograph-rafah


Running text_only:   1%|          | 1/99 [00:06<11:07,  6.81s/it]

  NLI [FC]: e=0.199  c=0.056  src=https://fullfact.org/news/iran-missile-drone-attac
  [FC REJECT] off-topic article: https://www.snopes.com/news/2024/01/22/israel-hamas-what-is-
  [FC SKIP] only 0 matches (need 1): https://www.snopes.com/fact-check/close-up-bees-face/
  [FC SKIP] only 0 matches (need 1): https://www.snopes.com/news/2022/04/08/mayim-bialik-allegati
  [FC SKIP] only 0 matches (need 1): https://www.factcheck.org/2012/10/factchecking-the-hofstra-d
  [FC SKIP] only 0 matches (need 1): https://www.factcheck.org/2017/05/trump-not-deporting-native
  [FC SKIP] only 0 matches (need 1): https://www.snopes.com/author/jordan/?pagenum=86


Running text_only:   2%|▏         | 2/99 [00:14<12:13,  7.56s/it]

...

  [FC SKIP] only 0 matches (need 1): https://www.politifact.com/factchecks/2011/sep/05/barack-oba
  [FC SKIP] only 0 matches (need 1): https://www.politifact.com/article/2010/jan/13/announcing-po
  [FC SKIP] only 0 matches (need 1): https://www.politifact.com/factchecks/2011/oct/06/critics-wa
  NLI [FC]: e=0.013  c=0.003  src=https://www.politifact.com/article/2012/oct/19/fac
  [FC REJECT] off-topic article: https://www.politifact.com/factchecks/2011/jan/31/sameh-shou
  [FC REJECT] off-topic article: https://www.politifact.com/factchecks/?page=709&
  [FC SKIP] only 0 matches (need 1): https://www.politifact.com/factchecks/2014/feb/11/hugh-fitzs


Running text_only:  99%|█████████▉| 98/99 [11:29<00:07,  7.66s/it]

  [FC REJECT] off-topic article: https://www.politifact.com/factchecks/2021/jun/04/tiktok-pos
  [FC REJECT] off-topic article: https://www.politifact.com/factchecks/2025/aug/07/tiktok-pos
  [FALLBACK] Using embedding similarity


Running text_only: 100%|██████████| 99/99 [11:42<00:00,  7.10s/it]

  NLI [FC]: e=0.028  c=0.367  src=https://www.politifact.com/factchecks/2020/jun/03/
  [FC REJECT] off-topic article: https://www.snopes.com/news/2026/03/13/katie-johnson-arreste
  [FC REJECT] off-topic article: https://www.politifact.com/factchecks/2024/jul/12/threads-po
MODE: text_only
  mode                 text_only
  n_total              99
  n_valid              99
  n_error              0
  accuracy             0.4949
  precision_macro      0.4974
  recall_macro         0.4998
  f1_macro             0.348

Prediction distribution:
pred
FAKE    97
REAL     2
Name: count, dtype: int64

              precision    recall  f1-score   support

        FAKE       0.49      0.98      0.66        49
        REAL       0.50      0.02      0.04        50

    accuracy                           0.49        99
   macro avg       0.50      0.50      0.35        99
weighted avg       0.50      0.49      0.34        99


Confusion matrix (rows=true, cols=pred):
      FAKE  REAL
FAKE    48     1

EVALUATION LOOP (CLEAN)

In [82]:
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

LABEL_ORDER = ["FAKE", "MISLEADING", "REAL"]

def evaluate_mode_v2(
    dataset,
    mode="text_only",
    top_k_text=4,
    use_fc=True,
    per_sample_timeout=90,
    verbose=True,
):
    LABEL_ORDER = ["FAKE", "REAL"]

    results = []
    for item in tqdm(dataset, desc=f"Running {mode}"):
        try:
            row = run_one_sample_ablation(
                item, mode=mode, top_k_text=top_k_text,
                use_fc=use_fc, binary_dataset=True,
            )
        except Exception as e:
            row = {"claim": item.get("claim",""), "true": item.get("label",""),
                   "pred": "ERROR", "mode": mode, "confidence": 0.0}
        results.append(row)

    df    = pd.DataFrame(results)
    valid = df[df["pred"].astype(str) != "ERROR"].copy()
    y_true = valid["true"].astype(str)
    y_pred = valid["pred"].astype(str)

    acc         = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABEL_ORDER, average=None, zero_division=0
    )
    summary = {
        "mode":            mode,
        "n_total":         len(df),
        "n_valid":         len(valid),
        "n_error":         len(df) - len(valid),
        "accuracy":        round(acc, 4),
        "precision_macro": round(float(p.mean()), 4),
        "recall_macro":    round(float(r.mean()), 4),
        "f1_macro":        round(float(f1.mean()), 4),
    }
    cm     = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    report = classification_report(y_true, y_pred, labels=LABEL_ORDER, zero_division=0)

    if verbose:
        print("=" * 60)
        print(f"MODE: {mode}")
        for k, v in summary.items():
            print(f"  {k:<20} {v}")
        print(f"\nPrediction distribution:\n{valid['pred'].value_counts()}")
        print(f"\n{report}")
        print("\nConfusion matrix (rows=true, cols=pred):")
        print(pd.DataFrame(cm, index=LABEL_ORDER, columns=LABEL_ORDER))
        print("\nPer-class correct/total:")
        for lab in LABEL_ORDER:
            sub = valid[valid["true"] == lab]
            correct = (sub["pred"] == lab).sum()
            print(f"  {lab:<12} {correct}/{len(sub)}  ({100*correct/max(len(sub),1):.0f}%)")

    return df, summary, cm, report


# ── RUN ALL THREE MODES ───────────────────────────────────────────────────────
modes       = ["text_only", "text_clip", "text_clip_ocr"]
summaries   = []
all_outputs = {}

for m in modes:
    df_mode, summary, cm, report = evaluate_mode_v2(
        dataset=final_eval_dataset,
        mode=m,
        top_k_text=4,
        use_fc=True,
        per_sample_timeout=90,
        verbose=True,
    )
    path = f"/content/ablation_v2_{m}.csv"
    df_mode.to_csv(path, index=False)
    print(f"  Saved: {path}")
    all_outputs[m] = {"df": df_mode, "summary": summary}
    summaries.append(summary)

summary_df = (
    pd.DataFrame(summaries)
    .sort_values("f1_macro", ascending=False)
    .reset_index(drop=True)
)
print("\n" + "="*60)
print("ABLATION SUMMARY (sorted by macro-F1)")
print(summary_df[["mode","n_valid","accuracy","precision_macro",
                   "recall_macro","f1_macro"]].to_string(index=False))
summary_df.to_csv("/content/ablation_v2_summary.csv", index=False)
print("\nSaved: /content/ablation_v2_summary.csv")

# ── SAVE TO GOOGLE DRIVE SO FILES SURVIVE RESTART ────────────────────────────
import shutil
files_to_save = [
    "/content/ablation_v2_text_only.csv",
    "/content/ablation_v2_text_clip.csv",
    "/content/ablation_v2_text_clip_ocr.csv",
    "/content/ablation_v2_summary.csv",
]
for f in files_to_save:
    try:
        dest = f.replace("/content/", "/content/drive/MyDrive/")
        shutil.copy(f, dest)
        print(f"Saved to Drive: {dest}")
    except Exception as e:
        print(f"Failed: {f} — {e}")

Running text_only:   0%|          | 0/99 [00:00<?, ?it/s]

  [FC SKIP] only 0 matches (need 1): https://www.snopes.com/fact-check/hamas-wedding-girls/
  [FC SKIP] only 0 matches (need 1): https://fullfact.org/conflict/
  NLI [FC]: e=0.285  c=0.532  src=https://www.snopes.com/fact-check/photograph-rafah


Running text_only:   1%|          | 1/99 [00:21<35:06, 21.49s/it]

  NLI [FC]: e=0.199  c=0.056  src=https://fullfact.org/news/iran-missile-drone-attac
  [FC REJECT] off-topic article: https://www.snopes.com/news/2024/01/22/israel-hamas-what-is-
  [FC SKIP] only 0 matches (need 1): https://www.snopes.com/fact-check/close-up-bees-face/
  [FC SKIP] only 0 matches (need 1): https://www.snopes.com/news/2022/04/08/mayim-bialik-allegati
  [FC SKIP] only 0 matches (need 1): https://www.factcheck.org/2012/10/factchecking-the-hofstra-d
  [FC SKIP] only 0 matches (need 1): https://www.factcheck.org/2017/05/trump-not-deporting-native
  [FC SKIP] only 0 matches (need 1): https://www.snopes.com/author/jordan/?pagenum=86


Running text_only:   2%|▏         | 2/99 [00:39<31:31, 19.50s/it]

...

  [FC SKIP] only 0 matches (need 1): https://www.snopes.com/fact-check/close-up-bees-face/
  [FC SKIP] only 0 matches (need 1): https://www.snopes.com/news/2022/04/08/mayim-bialik-allegati
  [FC SKIP] only 0 matches (need 1): https://www.factcheck.org/2012/10/factchecking-the-hofstra-d
  [FC SKIP] only 0 matches (need 1): https://www.factcheck.org/2017/05/trump-not-deporting-native
  [FC SKIP] only 0 matches (need 1): https://www.snopes.com/author/jordan/?pagenum=86


Running text_clip:   2%|▏         | 2/99 [00:13<11:14,  6.96s/it]

  [FC SKIP] only 0 matches (need 1): https://www.politifact.com/article/2025/mar/31/anonymous-x-a
  NLI [FC]: e=0.001  c=0.004  src=https://www.politifact.com/article/2023/oct/23/how
  [FC REJECT] off-topic article: https://www.snopes.com/fact-check/x-post-2023-cole-allen/
  [FC REJECT] off-topic article: https://www.snopes.com/news/2023/11/13/video-palestinian-man


Running text_clip:   2%|▏         | 2/99 [00:14<11:58,  7.41s/it]


KeyboardInterrupt: 

In [88]:
item = final_eval_dataset[0]
row = run_one_sample_ablation(
    item,
    mode="text_only",
    top_k_text=4,
    use_fc=True,
    binary_dataset=True,
)
print("Pred:", row["pred"])
print("True:", row["true"])
print("Confidence:", row["confidence"])

  [FC SKIP] only 0 matches (need 1): https://www.snopes.com/fact-check/hamas-wedding-girls/
  [FC SKIP] only 0 matches (need 1): https://fullfact.org/conflict/
  NLI [FC]: e=0.285  c=0.532  src=https://www.snopes.com/fact-check/photograph-rafah
  NLI [FC]: e=0.199  c=0.056  src=https://fullfact.org/news/iran-missile-drone-attac
  [FC REJECT] off-topic article: https://www.snopes.com/news/2024/01/22/israel-hamas-what-is-
Pred: FAKE
True: FAKE
Confidence: 29.42


In [ ]:
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

LABEL_ORDER = ["FAKE", "MISLEADING", "REAL"]

def evaluate_mode_v2(
    dataset,
    mode="text_only",
    top_k_text=4,
    use_fc=True,
    per_sample_timeout=90,   # DeBERTa is slower than distilbert — give it time
    verbose=True,
):
    LABEL_ORDER = ["FAKE", "REAL"]   # binary — no MISLEADING

    results = []
    for item in tqdm(dataset, desc=f"Running {mode}"):
        try:
            row = run_one_sample_ablation(
                item, mode=mode, top_k_text=top_k_text,
                use_fc=use_fc, binary_dataset=True,
            )
        except Exception as e:
            row = {"claim": item.get("claim",""), "true": item.get("label",""),
                   "pred": "ERROR", "mode": mode, "confidence": 0.0}
        results.append(row)

    df    = pd.DataFrame(results)
    valid = df[df["pred"].astype(str) != "ERROR"].copy()
    y_true = valid["true"].astype(str)
    y_pred = valid["pred"].astype(str)

    acc         = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABEL_ORDER, average=None, zero_division=0
    )
    summary = {
        "mode":            mode,
        "n_total":         len(df),
        "n_valid":         len(valid),
        "n_error":         len(df) - len(valid),
        "accuracy":        round(acc, 4),
        "precision_macro": round(float(p.mean()), 4),
        "recall_macro":    round(float(r.mean()), 4),
        "f1_macro":        round(float(f1.mean()), 4),
    }
    cm     = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    report = classification_report(y_true, y_pred, labels=LABEL_ORDER, zero_division=0)

    if verbose:
        print("=" * 60)
        print(f"MODE: {mode}")
        for k, v in summary.items():
            print(f"  {k:<20} {v}")
        print(f"\nPrediction distribution:\n{valid['pred'].value_counts()}")
        print(f"\n{report}")
        print("\nConfusion matrix (rows=true, cols=pred):")
        print(pd.DataFrame(cm, index=LABEL_ORDER, columns=LABEL_ORDER))
        print("\nPer-class correct/total:")
        for lab in LABEL_ORDER:
            sub = valid[valid["true"] == lab]
            correct = (sub["pred"] == lab).sum()
            print(f"  {lab:<12} {correct}/{len(sub)}  ({100*correct/max(len(sub),1):.0f}%)")

    return df, summary, cm, report


# ── NOW RUN IT ────────────────────────────────────────────────────────────────
modes     = ["text_only", "text_clip", "text_clip_ocr"]
summaries = []
all_outputs = {}

for m in modes:
    df_mode, summary, cm, report = evaluate_mode_v2(
        dataset=final_eval_dataset,
        mode=m,
        top_k_text=4,
        use_fc=True,
        per_sample_timeout=90,
        verbose=True,
    )
    path = f"/content/ablation_v2_{m}.csv"
    df_mode.to_csv(path, index=False)
    print(f"  Saved: {path}")
    all_outputs[m] = {"df": df_mode, "summary": summary}
    summaries.append(summary)

summary_df = (
    pd.DataFrame(summaries)
    .sort_values("f1_macro", ascending=False)
    .reset_index(drop=True)
)
print("\n" + "="*60)
print("ABLATION SUMMARY (sorted by macro-F1)")
print(summary_df[["mode","n_valid","accuracy","precision_macro",
                   "recall_macro","f1_macro"]].to_string(index=False))
summary_df.to_csv("/content/ablation_v2_summary.csv", index=False)
print("\nSaved: /content/ablation_v2_summary.csv")

Running text_only:   0%|          | 0/99 [00:00<?, ?it/s]

  [FC SKIP] only 0 matches (need 2): https://www.snopes.com/fact-check/hamas-wedding-girls/


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  [FC SKIP] only 1 matches (need 2): https://www.snopes.com/news/2023/10/13/idf-white-phosphorus-
  [FC SKIP] only 0 matches (need 2): https://fullfact.org/conflict/
  [FC SKIP] only 0 matches (need 2): https://www.snopes.com/fact-check/ai-photo-missiles-tel-aviv
  NLI [FC]: e=0.285  c=0.532  src=https://www.snopes.com/fact-check/photograph-rafah
  NLI [FC]: e=0.199  c=0.056  src=https://fullfact.org/news/iran-missile-drone-attac
  NLI [FC]: e=0.676  c=0.026  src=https://www.snopes.com/fact-check/richard-engel-is


Running text_only:   1%|          | 1/99 [00:08<13:35,  8.33s/it]

...

  NLI     : e=0.712  c=0.003  src=https://www.nytimes.com/2024/12/26/opinion/blake-l
  NLI     : e=0.003  c=0.001  src=https://www.bbc.com/news/articles/c9q7pxnr4g2o
  [FC SKIP] only 1 matches (need 2): https://www.politifact.com/factchecks/2025/aug/07/tiktok-pos
  [FC SKIP] only 0 matches (need 2): https://www.politifact.com/factchecks/2014/feb/11/hugh-fitzs
  [FC SKIP] only 1 matches (need 2): https://www.politifact.com/factchecks/2021/jun/04/tiktok-pos
  NLI     : e=0.135  c=0.006  src=https://www.foxnews.com/entertainment/michelle-pfe
  NLI     : e=0.003  c=0.010  src=https://www.elle.com/culture/movies-tv/a63196619/t
  NLI     : e=0.699  c=0.005  src=https://variety.com/2024/tv/news/michelle-pfeiffer


Running text_clip_ocr:  99%|█████████▉| 98/99 [09:03<00:04,  4.38s/it]

  NLI     : e=0.001  c=0.986  src=https://www.yahoo.com/entertainment/tv/articles/ma
  [FC SKIP] only 1 matches (need 2): https://www.politifact.com/factchecks/2024/jul/12/threads-po
  [FC SKIP] only 1 matches (need 2): https://www.politifact.com/truth-o-meter/promises/trumpomete
  [FC SKIP] only 1 matches (need 2): https://www.politifact.com/article/2023/may/12/ask-politifac
  [FC SKIP] only 1 matches (need 2): https://www.politifact.com/factchecks/2022/apr/28/facebook-p
  NLI     : e=0.497  c=0.025  src=https://www.foxnews.com/entertainment/jay-zs-sexua
  NLI     : e=0.751  c=0.015  src=https://www.tmz.com/2024/12/13/jay-z-diddy-rape-ac
  NLI [FC]: e=0.001  c=0.001  src=https://www.politifact.com/factchecks/2017/may/05/


Running text_clip_ocr: 100%|██████████| 99/99 [09:10<00:00,  5.56s/it]

  NLI [FC]: e=0.028  c=0.367  src=https://www.politifact.com/factchecks/2020/jun/03/
MODE: text_clip_ocr
  mode                 text_clip_ocr
  n_total              99
  n_valid              99
  n_error              0
  accuracy             0.5556
  precision_macro      0.766
  recall_macro         0.551
  f1_macro             0.4398

Prediction distribution:
pred
REAL    94
FAKE     5
Name: count, dtype: int64

              precision    recall  f1-score   support

        FAKE       1.00      0.10      0.19        49
        REAL       0.53      1.00      0.69        50

    accuracy                           0.56        99
   macro avg       0.77      0.55      0.44        99
weighted avg       0.76      0.56      0.44        99


Confusion matrix (rows=true, cols=pred):
      FAKE  REAL
FAKE     5    44
REAL     0    50

Per-class correct/total:
  FAKE         5/49  (10%)
  REAL         50/50  (100%)
  Saved: /content/ablation_v2_text_clip_ocr.csv

ABLATION SUMMARY (sorted by macr

In [ ]:
# Run in a separate cell to confirm GPU is active
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

False


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FULL ACCURACY EVALUATION — run this as one cell
# ═══════════════════════════════════════════════════════════════

import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from tqdm import tqdm
from collections import Counter

LABEL_ORDER = ["FAKE", "MISLEADING", "REAL"]


eval_text = [
    {"claim": x["claim"], "label": x["label"], "image_path": None}
    for x in dataset_100
]


df_result, summary, cm, report = evaluate_mode(
    eval_text,
    mode="text_only",
    top_k_text=2,
    per_sample_timeout=45,
    verbose=True,
)

print("\nConfusion matrix (rows=true, cols=pred):")
cm_df = pd.DataFrame(cm, index=LABEL_ORDER, columns=LABEL_ORDER)
print(cm_df)

print("\nPer-class correct/total:")
for lab in LABEL_ORDER:
    sub = df_result[df_result["true"] == lab]
    correct = (sub["pred"] == lab).sum()
    print(f"  {lab:<12} {correct}/{len(sub)}  "
          f"({100*correct/max(len(sub),1):.0f}%)")

df_result.to_csv("/content/eval_text_only_fixed.csv", index=False)
print("\nSaved: /content/eval_text_only_fixed.csv")

Running text_only:   0%|          | 0/100 [00:06<?, ?it/s]


KeyboardInterrupt: 

dataset

In [68]:
import json
import os
from collections import Counter

ROOT = "/content/drive/MyDrive/MyFakeNewsProject/IMFND_dataset/XFACTA_extracted"

def load_xfacta():
    dataset = []

    json_files = []

    for folder in ["fake_sample", "real_sample"]:
        json_files += [
            os.path.join(ROOT, folder, f)
            for f in os.listdir(os.path.join(ROOT, folder))
            if f.endswith(".json")
        ]

    for jf in json_files:
        with open(jf, "r") as f:
            data = json.load(f)

        for item in data:
            tweet = item.get("ooc_tweet", {})
            text = tweet.get("text", "")

            label_bool = tweet.get("metadata", {}).get("label", None)

            if label_bool is True:
                label = "REAL"
            elif label_bool is False:
                label = "FAKE"
            else:
                continue

            images = tweet.get("images", [])
            img_path = images[0] if len(images) > 0 else None

            if img_path and img_path.startswith("./"):
                img_path = os.path.join(ROOT, folder, img_path.replace("./", ""))

            dataset.append({
                "claim": text,
                "image_path": img_path,
                "label": label
            })

    return dataset
dataset = load_xfacta()

print("Total:", len(dataset))
print("Labels:", Counter([d["label"] for d in dataset]))

Total: 2005
Labels: Counter({'FAKE': 1005, 'REAL': 1000})


In [69]:
import random
from collections import defaultdict
import os
from PIL import Image

def sample_100(dataset, seed=42):
    random.seed(seed)

    grouped = defaultdict(list)
    for d in dataset:
        grouped[d["label"]].append(d)

    sample = []
    sample += random.sample(grouped["FAKE"], 50)
    sample += random.sample(grouped["REAL"], 50)

    random.shuffle(sample)
    return sample

small_dataset = sample_100(dataset)

print("Selected:", len(small_dataset))

Selected: 100


In [70]:
def verify_images(data):
    valid = []
    missing = []

    for d in data:
        img_path = d["image_path"]

        if img_path and os.path.exists(img_path):
            try:
                Image.open(img_path).verify()
                valid.append(d)
            except:
                missing.append(d)
        else:
            missing.append(d)

    return valid, missing


valid_data, missing_data = verify_images(small_dataset)

print("Valid samples:", len(valid_data))
print("Missing/Corrupt images:", len(missing_data))

Valid samples: 99
Missing/Corrupt images: 1


In [71]:
from collections import Counter

print("Label distribution (VALID ONLY):")
print(Counter([d["label"] for d in valid_data]))

Label distribution (VALID ONLY):
Counter({'REAL': 50, 'FAKE': 49})


In [72]:
final_eval_set = random.sample(
    [d for d in valid_data if d["label"] == "REAL"], 49
) + \
random.sample(
    [d for d in valid_data if d["label"] == "FAKE"], 49
)

In [73]:
final_dataset = valid_data

In [74]:
import json

with open("xfacta_100_eval.json", "w") as f:
    json.dump(final_dataset, f, indent=2)

In [75]:
final_eval_dataset = [
    {
        "claim": d["claim"],
        "image_path": d["image_path"],
        "label": d["label"]
    }
    for d in valid_data
]

In [76]:
sum([1 for d in final_eval_dataset if d["claim"] == ""])

1

In [ ]:
all_outputs, summary_df = run_ablation_suite(
    final_eval_dataset,
    top_k_text=2,

    per_sample_timeout=60
)

NameError: name 'evaluate_mode' is not defined

CREATE DATASET

In [ ]:

print("\n" + "="*65)
print("ABLATION RESULTS — MULTIMODAL DATASET")
print("="*65)
print(f"{'Mode':<18} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-"*65)
for _, row in summary_df.iterrows():
    print(f"{row['mode']:<18} {row['accuracy']:>10.4f} "
          f"{row['precision_macro']:>10.4f} "
          f"{row['recall_macro']:>10.4f} "
          f"{row['f1_macro']:>10.4f}")
print("="*65)

# ── compare against text-only baseline ──────────────────────────
print("\nComparison vs text-only baseline (no images):")
print(f"  text_only  (no images): accuracy=0.61, f1=0.61")
for _, row in summary_df.iterrows():
    delta_acc = row['accuracy'] - 0.61
    delta_f1  = row['f1_macro'] - 0.61
    sign_acc  = "+" if delta_acc >= 0 else ""
    sign_f1   = "+" if delta_f1  >= 0 else ""
    print(f"  {row['mode']:<18} (with images): "
          f"acc={sign_acc}{delta_acc:+.4f}  f1={sign_f1}{delta_f1:+.4f}")

# ── save everything ──────────────────────────────────────────────
summary_df.to_csv("/content/ablation_summary_multimodal.csv", index=False)
print("\nSaved: /content/ablation_summary_multimodal.csv")


ABLATION RESULTS — MULTIMODAL DATASET
Mode                 Accuracy  Precision     Recall         F1
-----------------------------------------------------------------
text_only              0.6300     0.6394     0.6298     0.6276
text_clip              0.6000     0.6059     0.5986     0.5863
text_clip_ocr          0.5900     0.5946     0.5885     0.5735

Comparison vs text-only baseline (no images):
  text_only  (no images): accuracy=0.61, f1=0.61
  text_only          (with images): acc=++0.0200  f1=++0.0176
  text_clip          (with images): acc=-0.0100  f1=-0.0237
  text_clip_ocr      (with images): acc=-0.0200  f1=-0.0365

Saved: /content/ablation_summary_multimodal.csv


Run full ablation

In [ ]:
def build_chapter4_tables(modes=None, k=3):
    if modes is None:
        modes = ["text_only", "text_clip", "text_clip_ocr"]

    all_summaries, all_best, all_worst = [], [], []

    for mode in modes:
        path = f"/content/ablation_{mode}_full_outputs.csv"
        if not os.path.exists(path):
            print(f"  Missing: {path}")
            continue
        df = pd.read_csv(path)

        y_true = df["true"].astype(str)
        y_pred = df["pred"].astype(str)
        acc = accuracy_score(y_true, y_pred)
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=LABEL_ORDER, average=None, zero_division=0
        )
        row = {"mode": mode, "n_valid": len(df), "accuracy": float(acc), "f1_macro": float(f1.mean())}
        for i, lab in enumerate(LABEL_ORDER):
            row[f"recall_{lab}"] = float(r[i])
        all_summaries.append(row)

        df["correct"] = y_pred == y_true
        use_cols = [c for c in ["claim", "true", "pred", "confidence", "explanation"] if c in df.columns]
        best = df[df["correct"]].sort_values("confidence", ascending=False).head(k)[use_cols].copy()
        worst = df[~df["correct"]].sort_values("confidence", ascending=False).head(k)[use_cols].copy()
        best["mode"] = worst["mode"] = mode
        all_best.append(best)
        all_worst.append(worst)

    summary_df = pd.DataFrame(all_summaries).sort_values("f1_macro", ascending=False).reset_index(drop=True)
    best_df = pd.concat(all_best, ignore_index=True) if all_best else pd.DataFrame()
    worst_df = pd.concat(all_worst, ignore_index=True) if all_worst else pd.DataFrame()

    summary_df.to_csv("/content/ch4_ablation_summary_tables.csv", index=False)
    best_df.to_csv("/content/ch4_ablation_best_examples.csv", index=False)
    worst_df.to_csv("/content/ch4_ablation_worst_examples.csv", index=False)

    print("=== Chapter 4 Summary ===")
    print(summary_df.to_string(index=False))
    return summary_df, best_df, worst_df


# ── Example usage ────────────────────────────────────────────
if __name__ == "__main__":
    eval_data = [
        {"claim": "NASA confirms water on Mars",    "label": "MISLEADING", "image_path": None},
        {"claim": "Smoking causes lung cancer",     "label": "REAL",       "image_path": None},
        {"claim": "The Earth is flat",              "label": "FAKE",       "image_path": None},
    ]

    all_outputs, summary_df = run_ablation_suite(eval_data, top_k_text=2, per_sample_timeout=45)
    print(summary_df)

In [ ]:
def best_worst_examples(df, mode, k=3):
    df = df[df["pred"].astype(str) != "ERROR"].copy()
    if len(df) == 0:
        return pd.DataFrame(), pd.DataFrame()
    df["correct"] = df["pred"].astype(str) == df["true"].astype(str)
    conf_col = "confidence" if "confidence" in df.columns else None
    use_cols = [c for c in [
        "claim", "true", "pred", "confidence",
        "evidence_texts", "explanation", "image_status", "clip_sim_raw", "ocr_len"
    ] if c in df.columns]
    # Best = correct & highest confidence
    best = df[df["correct"]].copy()
    if conf_col:
        best = best.sort_values("confidence", ascending=False)
    best = best.head(k)[use_cols].reset_index(drop=True)
    # Worst = incorrect & highest confidence
    worst = df[~df["correct"]].copy()
    if conf_col:
        worst = worst.sort_values("confidence", ascending=False)
    worst = worst.head(k)[use_cols].reset_index(drop=True)
    best["mode"] = mode
    worst["mode"] = mode
    return best, worst
all_summaries = []
all_best = []
all_worst = []
for mode in modes:
    path = f"/content/ablation_{mode}_full_outputs.csv"
    if not os.path.exists(path):
        print(f"Missing: {path}")
        continue
    df = pd.read_csv(path)
    all_summaries.append(make_summary_row(df, mode))
    best, worst = best_worst_examples(df, mode, k=3)
    all_best.append(best)
    all_worst.append(worst)
summary_df = pd.DataFrame(all_summaries).sort_values("f1_macro", ascending=False).reset_index(drop=True)
best_examples_df = pd.concat(all_best, ignore_index=True) if all_best else pd.DataFrame()
worst_examples_df = pd.concat(all_worst, ignore_index=True) if all_worst else pd.DataFrame()
# Save thesis artifacts
summary_path = "/content/ch4_ablation_summary_tables.csv"
best_path = "/content/ch4_ablation_best_examples.csv"
worst_path = "/content/ch4_ablation_worst_examples.csv"
summary_df.to_csv(summary_path, index=False)
best_examples_df.to_csv(best_path, index=False)
worst_examples_df.to_csv(worst_path, index=False)
print("=== Chapter 4 Summary Table ===")
display(summary_df)
print("\nSaved:")
print(summary_path)
print(best_path)
print(worst_path)
print("\n=== Best Examples (highest confidence, correct) ===")
display(best_examples_df.head(12))
print("\n=== Worst Examples (highest confidence, wrong) ===")
display(worst_examples_df.head(12))

=== Chapter 4 Summary Table ===


,mode,n_valid,accuracy,f1_macro,recall_FAKE,recall_MISLEADING,recall_REAL
0,text_only,3,0.333333,0.166667,0.0,1.0,0.0
1,text_clip,3,0.333333,0.166667,0.0,1.0,0.0
2,text_clip_ocr,3,0.333333,0.166667,0.0,1.0,0.0



Saved:
/content/ch4_ablation_summary_tables.csv
/content/ch4_ablation_best_examples.csv
/content/ch4_ablation_worst_examples.csv

=== Best Examples (highest confidence, correct) ===


,claim,true,pred,confidence,evidence_texts,explanation,image_status,clip_sim_raw,ocr_len,mode
0,NASA confirms water on Mars,MISLEADING,MISLEADING,0.00,"""This makes it sound like we announced that we...",Claim: NASA confirms water on Mars\n\nTop evid...,skipped,0.000000,0,text_only
1,NASA confirms water on Mars,MISLEADING,MISLEADING,15.42,"""This makes it sound like we announced that we...",Claim: NASA confirms water on Mars\n\nTop evid...,url_ok,0.233853,0,text_clip
2,NASA confirms water on Mars,MISLEADING,MISLEADING,15.42,"""This makes it sound like we announced that we...",Claim: NASA confirms water on Mars\n\nTop evid...,url_ok,0.233853,0,text_clip_ocr



=== Worst Examples (highest confidence, wrong) ===


,claim,true,pred,confidence,evidence_texts,explanation,image_status,clip_sim_raw,ocr_len,mode
0,Smoking causes lung cancer,REAL,MISLEADING,0.0,Cancer Mythbusters: Smoking and Lung Cancer Yo...,Claim: Smoking causes lung cancer\n\nTop evide...,skipped,0.0,0,text_only
1,The Earth is flat,FAKE,MISLEADING,0.0,"Experts told Reuters that the idea, which orig...",Claim: The Earth is flat\n\nTop evidence:\n1. ...,skipped,0.0,0,text_only
2,Smoking causes lung cancer,REAL,MISLEADING,12.5,Cancer Mythbusters: Smoking and Lung Cancer Yo...,Claim: Smoking causes lung cancer\n\nTop evide...,no_image,0.0,0,text_clip
3,The Earth is flat,FAKE,MISLEADING,12.5,"Experts told Reuters that the idea, which orig...",Claim: The Earth is flat\n\nTop evidence:\n1. ...,no_image,0.0,0,text_clip
4,Smoking causes lung cancer,REAL,MISLEADING,12.5,Cancer Mythbusters: Smoking and Lung Cancer Yo...,Claim: Smoking causes lung cancer\n\nTop evide...,no_image,0.0,0,text_clip_ocr
5,The Earth is flat,FAKE,MISLEADING,12.5,"Experts told Reuters that the idea, which orig...",Claim: The Earth is flat\n\nTop evidence:\n1. ...,no_image,0.0,0,text_clip_ocr


In [ ]:
# Save all outputs to Google Drive so they survive runtime restarts
import shutil

files_to_save = [
    "/content/ablation_v2_text_only.csv",
    "/content/ablation_v2_text_clip.csv",
    "/content/ablation_v2_text_clip_ocr.csv",
    "/content/ablation_v2_summary.csv",
]

for f in files_to_save:
    try:
        dest = f.replace("/content/", "/content/drive/MyDrive/")
        shutil.copy(f, dest)
        print(f"Saved to Drive: {dest}")
    except Exception as e:
        print(f"Failed: {f} — {e}")